# Cobertura eléctrica rural y rendimiento en Saber 11 (Colombia)
## Cuaderno ACA2 — Etapas 2 y 3 de la ruta del proyecto: ingeniería de características, modelo dimensional, KPIs, tablero, pruebas de hipótesis y batería de modelos

**Especialización en Analítica de Datos — CUN.** Proyecto de grado, Avance 2 (versión `v0.2`).

**Integrantes:** Diego Alejandro Prada Fernández · Juan David Sierra Laborde · Jhan Carlos Guerrero Silva
**Director:** Juan Pablo Morales Arias

Este cuaderno continúa el cuaderno `ACA1_etapas_1_y_2.ipynb` (usa el panel municipal que ese cuaderno genera y que está en el repositorio) y se ejecuta de principio a fin
sin intervención manual y con semilla fija. **Todas las cifras, tablas y figuras nuevas del documento de grado salen de aquí**: se guardan en
`results/tables/` (prefijo `A2_` y `cifras_aca2.json`), `results/figures/` (prefijo `A2_`), `results/tables/metricas_por_particion.csv`,
`results/experiments.csv` y `results/dashboard/tablero_aca2.html`.

| Parte del cuaderno | Etapa de la ruta | Objetivo específico |
|---|---|---|
| 1. Verificación del punto de partida y ajustes del ACA 1 | — | — |
| 2. Modelo dimensional (esquema estrella), KPIs y validación de totales | Etapa 3 · enfoque descriptivo | OE2 y OE4 (insumo) |
| 3. Ingeniería de características y selección de variables | Etapa 2 | OE3 (insumo) |
| 4. Pruebas de hipótesis H1 a H3 | Etapa 3 · rigor analítico | **OE2 · Analizar** |
| 5. Partición, validación y batería de modelos | Etapa 3 · modelado | **OE3 · Estimar** |
| 6. Modelo de datos panel H4 a H6 | Etapa 3 · rigor analítico | **OE3 · Estimar** |
| 7. Tablero de control | Etapa 3 · KPIs y tablero | OE4 (primera versión) |
| 8. Trazabilidad y salidas | — | — |

In [ ]:
# 0. Configuración del entorno -----------------------------------------------------------
# En Google Colab: clona el repositorio y entra en él (las celdas siguientes detectan la raíz automáticamente).
import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/alefer100master-stack/energia-rural-saber11.git"   # repositorio del proyecto (cámbielo si usa una copia)
if EN_COLAB and not os.path.exists("energia-rural-saber11"):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if EN_COLAB:
    os.chdir("energia-rural-saber11")

In [ ]:
import json, random, platform, time, datetime as dt, warnings, sqlite3, importlib
T0 = time.time()
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats
import sklearn
from sklearn.model_selection import StratifiedGroupKFold, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from sklearn.base import clone

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Semilla fija (reproducibilidad): la misma del ACA 1
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

# Rutas
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW_ICFES = ROOT / "data/raw/icfes"
RAW_UPME = ROOT / "data/raw/upme"
PROCESSED = ROOT / "data/processed"
DIMENSIONAL = PROCESSED / "modelo_dimensional"
FIG = ROOT / "results/figures"
TAB = ROOT / "results/tables"
DASH = ROOT / "results/dashboard"
for p in (PROCESSED, DIMENSIONAL, FIG, TAB, DASH):
    p.mkdir(parents=True, exist_ok=True)

# Estilo de figuras: el mismo del ACA 1 (marcas delgadas, rejilla recesiva, paleta azul–naranja–aqua)
C1, C2, C3, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.edgecolor": "#8a8984",
    "axes.titleweight": "bold", "legend.frameon": False,
})

CIFRAS = {}          # todas las cifras nuevas que cita el documento
def registrar(clave, valor):
    """Guarda una cifra calculada para que el documento la cite sin escribirla a mano."""
    if isinstance(valor, (np.floating, np.integer)):
        valor = valor.item()
    CIFRAS[clave] = valor
    return valor

def guardar_tabla(df, nombre, index=False):
    df.to_csv(TAB / f"{nombre}.csv", index=index, encoding="utf-8-sig")
    return df

def guardar_fig(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(FIG / f"{nombre}.svg", bbox_inches="tight", facecolor="white")
    return fig

try:
    display  # disponible en Jupyter / Colab
except NameError:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)

# XGBoost y LightGBM vienen instalados en Google Colab. Si el entorno no los tiene, la batería sigue sin ellos y lo deja registrado.
def disponible(modulo):
    try:
        importlib.import_module(modulo)
        return True
    except ImportError:
        return False
HAY_XGB, HAY_LGBM = disponible("xgboost"), disponible("lightgbm")
registrar("xgboost_disponible", HAY_XGB)
registrar("lightgbm_disponible", HAY_LGBM)

versiones = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
             "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__,
             "seaborn": sns.__version__, "sqlite": sqlite3.sqlite_version}
if HAY_XGB:
    import xgboost; versiones["xgboost"] = xgboost.__version__
if HAY_LGBM:
    import lightgbm; versiones["lightgbm"] = lightgbm.__version__
registrar("versiones", versiones)
print("Raíz del proyecto:", ROOT.name, "| Semilla:", SEED)
print(versiones)

## 1. Punto de partida y ajustes del ACA 1
El ACA 2 parte del panel municipal que produjo el ACA 1 (`data/processed/panel_municipal_rural_2019_2022.csv`). Antes de añadir contenido nuevo se
verifica que el panel sea exactamente el entregado (2.627 filas y ρ de Spearman = 0,301). Los ajustes pedidos en la retroalimentación del ACA 1
están en la sección 6 bis del cuaderno ACA1 (I de Moran, sensibilidad mediana frente a KNN e interacción ICEE × región) y aquí se aplican al modelado:
la imputación KNN va **dentro de un `Pipeline` que se ajusta solo con los datos de cada partición**, la sensibilidad a la imputación se repite sobre los
modelos finales y la interacción ICEE × región se prueba formalmente (H6).

In [ ]:
# float_precision="round_trip": lee los decimales exactamente como se guardaron. Con el lector por defecto algunos ICEE de 100 % quedan como
# 99,99999999999999 y eso cambia cómo Spearman trata los empates (el 37 % de los municipios tiene ICEE rural = 100 %).
panel = pd.read_csv(PROCESSED / "panel_municipal_rural_2019_2022.csv", dtype={"cod_mpio": str, "cod_depto": str}, float_precision="round_trip")
cifras_aca1 = json.load(open(TAB / "cifras.json", encoding="utf-8"))
rho_panel = stats.spearmanr(panel["icee_rural"], panel["punt_global"]).statistic
assert len(panel) == cifras_aca1["panel_filas"] == 2627, "El panel no coincide con el del ACA 1: ejecute primero ACA1_etapas_1_y_2.ipynb"
assert round(rho_panel, 3) == 0.301
registrar("panel_filas", len(panel))
registrar("panel_municipios", int(panel["cod_mpio"].nunique()))
print(f"Panel del ACA 1 verificado: {len(panel):,} filas, {panel['cod_mpio'].nunique()} municipios, ρ = {rho_panel:.3f}")

## 2. Etapa 3 · Modelo dimensional (esquema estrella)
El enfoque del proyecto es descriptivo y diagnóstico, así que la etapa 3 incluye un **modelo dimensional**: tablas de **hechos** (lo que se mide) rodeadas
de tablas de **dimensiones** (por qué se filtra o agrupa). Se guarda en una base SQLite (viene con Python, no requiere instalar nada) y los KPIs se
calculan con consultas SQL sobre ella. El tablero de la parte 7 lee de este modelo.

| Tabla | Tipo | Grano (una fila es…) | Medidas o atributos |
|---|---|---|---|
| `hechos_saber11` | Hechos | un estudiante evaluado | puntaje global y por área; hogar (internet, computador, estrato, educación de la madre); colegio oficial |
| `hechos_cobertura` | Hechos | un municipio en un año | viviendas rurales totales, con servicio y sin servicio; ICEE rural, urbano y total |
| `dim_municipio` | Dimensión | un municipio | nombre, departamento, región |
| `dim_tiempo` | Dimensión | un año | periodo del ICFES, etapa frente a la pandemia |
| `dim_zona` | Dimensión | una zona | rural o urbana |
| `dim_banda_icee` | Dimensión | una banda de cobertura | límites de la banda (70 % y 95 %) |

### 2.1 Reconstrucción de las tablas de hechos desde los datos crudos
Se aplican sobre las copias fechadas (verificadas con SHA-256 en el ACA 1) **las mismas reglas del ETL del ACA 1**, para que el modelo dimensional
nazca de la fuente y no del panel ya agregado.

In [ ]:
FUENTES_ICFES = ["saber11_kgxf-xxbe_periodo20194_descarga2026-10-15.csv.gz",
                 "saber11_rnvb-vnyh_periodo20204_descarga2026-10-15.csv.gz",
                 "saber11_kgxf-xxbe_periodo20224_descarga2026-10-15.csv.gz"]
icfes_raw = pd.concat([pd.read_csv(RAW_ICFES / f, dtype=str) for f in FUENTES_ICFES], ignore_index=True)
icfes_raw["anio"] = pd.to_numeric(icfes_raw["periodo"].str[:4], errors="coerce").astype(int)
print(f"Registros crudos del ICFES: {len(icfes_raw):,}")

def etl_saber11(crudo):
    """Las nueve reglas del ETL del ACA 1 (Tabla 7 del documento), en el mismo orden."""
    d = crudo.drop_duplicates(keep="first")                                        # 1. duplicados exactos
    d = d.drop_duplicates(subset=["estu_consecutivo"], keep="first")               # 2. un registro por evaluado
    d = d[d["estu_estadoinvestigacion"] == "PUBLICAR"]                             # 3. resultados válidos
    d = d[d["cole_calendario"] == "A"]                                             # 4. calendario A
    d = d[~d["cole_jornada"].isin(["SABATINA", "NOCHE"])]                          # 5. sin jornadas de adultos
    d = d[d["cole_area_ubicacion"].isin(["RURAL", "URBANO"]) & d["cole_cod_mcpio_ubicacion"].notna()]   # 6. zona y municipio
    d = d.copy()
    for c in ["punt_lectura_critica", "punt_matematicas", "punt_c_naturales", "punt_sociales_ciudadanas", "punt_ingles", "punt_global"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d[d["punt_global"].notna()]                                                # 7. puntaje global válido (8 solo verifica)
    d["cod_mpio"] = d["cole_cod_mcpio_ubicacion"].str.strip().str.zfill(5)        # 9. llaves y categorías
    d["rural"] = (d["cole_area_ubicacion"] == "RURAL").astype(int)
    d["oficial"] = (d["cole_naturaleza"] == "OFICIAL").astype(int)
    d["mujer"] = d["estu_genero"].map({"F": 1, "M": 0})
    d["internet"] = d["fami_tieneinternet"].map({"Si": 1, "No": 0})
    d["computador"] = d["fami_tienecomputador"].map({"Si": 1, "No": 0})
    estrato = d["fami_estratovivienda"].map({"Sin Estrato": 0, "Estrato 1": 1, "Estrato 2": 2, "Estrato 3": 3,
                                             "Estrato 4": 4, "Estrato 5": 5, "Estrato 6": 6})
    d["estrato_0a2"] = np.where(estrato.isna(), np.nan, (estrato <= 2).astype(float))
    media_o_mas = {"Secundaria (Bachillerato) completa", "Técnica o tecnológica incompleta", "Técnica o tecnológica completa",
                   "Educación profesional incompleta", "Educación profesional completa", "Postgrado"}
    validas = media_o_mas | {"Ninguno", "Primaria incompleta", "Primaria completa", "Secundaria (Bachillerato) incompleta"}
    d["madre_media_o_mas"] = np.where(d["fami_educacionmadre"].isin(validas), d["fami_educacionmadre"].isin(media_o_mas).astype(float), np.nan)
    return d

est = etl_saber11(icfes_raw)
assert len(est) == cifras_aca1["icfes_filas_limpias"], "El ETL no reproduce el conteo del ACA 1"
print(f"Estudiantes tras el ETL: {len(est):,} (igual al ACA 1)")

# ICEE: las seis reglas del ETL del ACA 1 (Tabla 9 del documento)
f_icee = RAW_UPME / "Anexo1_Calculo_ICEE_2019_2022_descarga2026-10-15.xlsx"
icee_1922 = pd.read_excel(f_icee, sheet_name="ICEE Municipal con Transf.", header=1)
icee_23 = pd.read_excel(RAW_UPME / "Anexo_ICEE_2023_dpto_mpio_descarga2026-10-15.xlsx", sheet_name="ICEE_MPIO", dtype={"Cod. DANE": str})
icee_24 = pd.read_excel(RAW_UPME / "Anexo_ICEE_2024_dpto_mpio_descarga2026-10-15.xlsx", sheet_name="ICEE_MPIO", dtype={"Cod. DANE": str})
a = icee_1922.rename(columns={"Divipola": "cod", "Año": "anio", "Departamento": "departamento", "Municipio": "municipio",
                              "ICEE Rural": "icee_rural", "ICEE Urbano": "icee_urbano", "ICEE": "icee_total",
                              "Viviendas totales - VT Rurales": "vt_rural", "Viviendas totales - VT": "vt_total"})
b = pd.concat([icee_23, icee_24]).rename(columns={"Cod. DANE": "cod", "Vigencia": "anio", "Departamento": "departamento", "Municipio": "municipio",
                                                  "ICEE Rural": "icee_rural", "ICEE Urbano": "icee_urbano", "ICEE Total": "icee_total",
                                                  "VT Rural": "vt_rural", "VT": "vt_total"})
COLS = ["cod", "anio", "departamento", "municipio", "icee_rural", "icee_urbano", "icee_total", "vt_rural", "vt_total"]
cob = pd.concat([a[COLS], b[COLS]], ignore_index=True)
cob["cod_mpio"] = cob["cod"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(5)
cob["anio"] = pd.to_numeric(cob["anio"]).astype(int)
for c in ["icee_rural", "icee_urbano", "icee_total"]:
    cob[c] = pd.to_numeric(cob[c], errors="coerce")
    escala_0_1 = cob.groupby("anio")[c].transform("max") <= 1.0001
    cob.loc[escala_0_1, c] = cob.loc[escala_0_1, c] * 100
    cob[c] = cob[c].clip(upper=100)
cob = cob.drop_duplicates(["cod_mpio", "anio"])
for c in ["vt_rural", "vt_total"]:
    cob[c] = pd.to_numeric(cob[c], errors="coerce")
cob["vss_rural"] = (cob["vt_rural"] * (1 - cob["icee_rural"] / 100)).round()
cob = cob[cob["icee_rural"].notna() & (cob["vt_rural"] > 0)].copy()
cob["vcs_rural"] = cob["vt_rural"] - cob["vss_rural"]
assert len(cob) == 6732
print(f"Filas de cobertura tras el ETL: {len(cob):,} (igual al ACA 1)")

### 2.2 Construcción de las dimensiones y los hechos

In [ ]:
REGION = {**{d: "Caribe" for d in ["08", "13", "20", "23", "44", "47", "70", "88"]},
          **{d: "Andina" for d in ["05", "11", "15", "17", "25", "41", "54", "63", "66", "68", "73"]},
          **{d: "Pacífica" for d in ["19", "27", "52", "76"]},
          **{d: "Orinoquía" for d in ["50", "81", "85", "99"]},
          **{d: "Amazonía" for d in ["18", "86", "91", "94", "95", "97"]}}
BANDAS = [(1, "Baja (<70 %)", 0, 70), (2, "Media (70–95 %)", 70, 95), (3, "Alta (≥95 %)", 95, 100.01)]

def id_banda(icee):
    return np.select([icee < 70, icee < 95], [1, 2], default=3)

# Dimensión municipio: nombres oficiales de la UPME; si el municipio no está en la UPME, el nombre del ICFES
nom_upme = cob.sort_values("anio").groupby("cod_mpio")[["municipio", "departamento"]].last()
nom_icfes = (est.groupby("cod_mpio")[["cole_mcpio_ubicacion", "cole_depto_ubicacion"]].agg(lambda s: s.mode().iat[0])
               .rename(columns={"cole_mcpio_ubicacion": "municipio", "cole_depto_ubicacion": "departamento"}))
dim_municipio = pd.concat([nom_upme, nom_icfes[~nom_icfes.index.isin(nom_upme.index)]]).reset_index()
dim_municipio["id_departamento"] = dim_municipio["cod_mpio"].str[:2]
dim_municipio["region"] = dim_municipio["id_departamento"].map(REGION)
dim_municipio = dim_municipio.rename(columns={"cod_mpio": "id_municipio"})[["id_municipio", "municipio", "id_departamento", "departamento", "region"]]

dim_tiempo = pd.DataFrame({"anio": range(2019, 2025)})
dim_tiempo["periodo_icfes"] = dim_tiempo["anio"].map({2019: "20194", 2020: "20204", 2022: "20224"})
dim_tiempo["etapa_pandemia"] = dim_tiempo["anio"].map(lambda a: "Prepandemia" if a == 2019 else ("Pandemia" if a in (2020, 2021) else "Pospandemia"))
dim_tiempo["tiene_saber11"] = dim_tiempo["periodo_icfes"].notna().astype(int)
dim_zona = pd.DataFrame({"id_zona": [0, 1], "zona": ["Urbana", "Rural"]})
dim_banda = pd.DataFrame(BANDAS, columns=["id_banda", "banda", "limite_inferior", "limite_superior"])

hechos_saber11 = est.rename(columns={"estu_consecutivo": "id_evaluado", "cod_mpio": "id_municipio", "rural": "id_zona",
                                     "cole_cod_dane_sede": "id_sede", "punt_lectura_critica": "punt_lectura",
                                     "punt_c_naturales": "punt_naturales", "punt_sociales_ciudadanas": "punt_sociales",
                                     "madre_media_o_mas": "madre_media"})[
    ["id_evaluado", "id_municipio", "anio", "id_zona", "id_sede", "oficial", "mujer", "estrato_0a2", "madre_media", "internet",
     "computador", "punt_global", "punt_lectura", "punt_matematicas", "punt_naturales", "punt_sociales", "punt_ingles"]]
hechos_cobertura = cob.rename(columns={"cod_mpio": "id_municipio"})[
    ["id_municipio", "anio", "vt_rural", "vcs_rural", "vss_rural", "vt_total", "icee_rural", "icee_urbano", "icee_total"]].copy()
hechos_cobertura["id_banda"] = id_banda(hechos_cobertura["icee_rural"])

# Base SQLite con llaves primarias y foráneas
DB = PROCESSED / "modelo_dimensional.sqlite"
if DB.exists():
    DB.unlink()
con = sqlite3.connect(DB)
con.executescript("""
CREATE TABLE dim_municipio (id_municipio TEXT PRIMARY KEY, municipio TEXT, id_departamento TEXT, departamento TEXT, region TEXT);
CREATE TABLE dim_tiempo (anio INTEGER PRIMARY KEY, periodo_icfes TEXT, etapa_pandemia TEXT, tiene_saber11 INTEGER);
CREATE TABLE dim_zona (id_zona INTEGER PRIMARY KEY, zona TEXT);
CREATE TABLE dim_banda_icee (id_banda INTEGER PRIMARY KEY, banda TEXT, limite_inferior REAL, limite_superior REAL);
CREATE TABLE hechos_saber11 (id_evaluado TEXT PRIMARY KEY, id_municipio TEXT REFERENCES dim_municipio, anio INTEGER REFERENCES dim_tiempo,
  id_zona INTEGER REFERENCES dim_zona, id_sede TEXT, oficial INTEGER, mujer REAL, estrato_0a2 REAL, madre_media REAL, internet REAL,
  computador REAL, punt_global REAL, punt_lectura REAL, punt_matematicas REAL, punt_naturales REAL, punt_sociales REAL, punt_ingles REAL);
CREATE TABLE hechos_cobertura (id_municipio TEXT REFERENCES dim_municipio, anio INTEGER REFERENCES dim_tiempo, vt_rural REAL, vcs_rural REAL,
  vss_rural REAL, vt_total REAL, icee_rural REAL, icee_urbano REAL, icee_total REAL, id_banda INTEGER REFERENCES dim_banda_icee,
  PRIMARY KEY (id_municipio, anio));
""")
for nombre, tabla in [("dim_municipio", dim_municipio), ("dim_tiempo", dim_tiempo), ("dim_zona", dim_zona), ("dim_banda_icee", dim_banda),
                      ("hechos_saber11", hechos_saber11), ("hechos_cobertura", hechos_cobertura)]:
    tabla.to_sql(nombre, con, if_exists="append", index=False, chunksize=50_000)
con.execute("CREATE INDEX ix_saber_mpio_anio ON hechos_saber11 (id_municipio, anio, id_zona)")
con.commit()

# Vista del panel rural: la misma unidad de análisis del ACA 1, calculada con SQL sobre el modelo
con.executescript("""
CREATE VIEW vista_panel_rural AS
WITH rural AS (
  SELECT id_municipio, anio, COUNT(*) AS n_estudiantes, AVG(punt_global) AS punt_global
  FROM hechos_saber11 WHERE id_zona = 1 GROUP BY id_municipio, anio HAVING COUNT(*) >= 10),
urbano AS (
  SELECT id_municipio, anio, COUNT(*) AS n_estudiantes_urbano, AVG(punt_global) AS punt_global_urbano
  FROM hechos_saber11 WHERE id_zona = 0 GROUP BY id_municipio, anio)
SELECT r.id_municipio, m.municipio, m.departamento, m.region, r.anio, t.etapa_pandemia, r.n_estudiantes, r.punt_global,
       u.n_estudiantes_urbano, u.punt_global_urbano, c.icee_rural, c.icee_urbano, c.vt_rural, c.vss_rural, b.id_banda, b.banda
FROM rural r
JOIN hechos_cobertura c ON c.id_municipio = r.id_municipio AND c.anio = r.anio
JOIN dim_municipio m ON m.id_municipio = r.id_municipio
JOIN dim_tiempo t ON t.anio = r.anio
JOIN dim_banda_icee b ON b.id_banda = c.id_banda
LEFT JOIN urbano u ON u.id_municipio = r.id_municipio AND u.anio = r.anio;
""")
def sql(q):
    return pd.read_sql_query(q, con)

conteo_dim = sql("""SELECT 'dim_municipio' AS tabla, COUNT(*) AS filas FROM dim_municipio UNION ALL
                    SELECT 'dim_tiempo', COUNT(*) FROM dim_tiempo UNION ALL SELECT 'dim_zona', COUNT(*) FROM dim_zona UNION ALL
                    SELECT 'dim_banda_icee', COUNT(*) FROM dim_banda_icee UNION ALL
                    SELECT 'hechos_saber11', COUNT(*) FROM hechos_saber11 UNION ALL
                    SELECT 'hechos_cobertura', COUNT(*) FROM hechos_cobertura UNION ALL
                    SELECT 'vista_panel_rural', COUNT(*) FROM vista_panel_rural""")
GRANO = {"dim_municipio": "Un municipio", "dim_tiempo": "Un año (2019–2024)", "dim_zona": "Una zona (rural o urbana)",
         "dim_banda_icee": "Una banda de cobertura", "hechos_saber11": "Un estudiante evaluado", "hechos_cobertura": "Un municipio en un año",
         "vista_panel_rural": "Un municipio en un año, zona rural (≥10 estudiantes)"}
LLAVES = {"dim_municipio": "id_municipio (PK)", "dim_tiempo": "anio (PK)", "dim_zona": "id_zona (PK)", "dim_banda_icee": "id_banda (PK)",
          "hechos_saber11": "id_evaluado (PK); id_municipio, anio, id_zona (FK)", "hechos_cobertura": "id_municipio + anio (PK); id_banda (FK)",
          "vista_panel_rural": "Une los dos hechos por id_municipio + anio"}
conteo_dim["Tipo"] = conteo_dim["tabla"].map(lambda t: "Hechos" if t.startswith("hechos") else ("Vista" if t.startswith("vista") else "Dimensión"))
conteo_dim["Grano"] = conteo_dim["tabla"].map(GRANO)
conteo_dim["Llaves"] = conteo_dim["tabla"].map(LLAVES)
conteo_dim = conteo_dim.rename(columns={"tabla": "Tabla", "filas": "Filas"})[["Tabla", "Tipo", "Grano", "Llaves", "Filas"]]
guardar_tabla(conteo_dim, "A2_T01_modelo_dimensional")
registrar("modelo_dimensional", conteo_dim.to_dict(orient="records"))

# Copias livianas en CSV (las dimensiones, la cobertura y la vista del panel); la base completa se regenera con este cuaderno
for nombre in ["dim_municipio", "dim_tiempo", "dim_zona", "dim_banda_icee", "hechos_cobertura", "vista_panel_rural"]:
    sql(f"SELECT * FROM {nombre}").to_csv(DIMENSIONAL / f"{nombre}.csv", index=False, encoding="utf-8")
display(conteo_dim)

In [ ]:
# Diagrama del esquema estrella
fig, ax = plt.subplots(figsize=(10, 5.6))
ax.set_xlim(0, 10); ax.set_ylim(0, 6.4); ax.axis("off"); ax.grid(False)
def caja(x, y, titulo, campos, color, w=2.6):
    h = 0.5 + 0.27 * len(campos) + 0.12
    ax.add_patch(matplotlib.patches.FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.02", fc="white", ec=color, lw=1.6, zorder=2))
    ax.add_patch(matplotlib.patches.Rectangle((x - w / 2, y + h / 2 - 0.4), w, 0.4, fc=color, ec=color, zorder=3))
    ax.text(x, y + h / 2 - 0.2, titulo, ha="center", va="center", color="white", fontsize=9, fontweight="bold", zorder=4)
    for i, c in enumerate(campos):
        ax.text(x - w / 2 + 0.12, y + h / 2 - 0.62 - 0.27 * i, c, fontsize=7.5, va="center", color="#1d2433", zorder=4)
    return {"x": x, "y": y, "w": w, "h": h, "arriba": y + h / 2, "abajo": y - h / 2, "izq": x - w / 2, "der": x + w / 2}
hs = caja(3.0, 3.2, "hechos_saber11", ["id_evaluado (PK)", "id_municipio (FK)", "anio (FK)", "id_zona (FK)", "punt_global y 5 áreas",
                                         "internet · computador", "estrato_0a2 · madre_media", "oficial · mujer"], C1, w=2.5)
hc = caja(7.0, 3.2, "hechos_cobertura", ["id_municipio + anio (PK)", "id_banda (FK)", "vt_rural · vcs_rural", "vss_rural · vt_total",
                                          "icee_rural · icee_urbano", "icee_total"], C2, w=2.5)
dm = caja(5.0, 5.75, "dim_municipio", ["id_municipio · municipio", "departamento · región"], GRIS, w=2.6)
dt_ = caja(5.0, 0.6, "dim_tiempo", ["anio · periodo_icfes", "etapa_pandemia"], GRIS, w=2.6)
dz = caja(0.85, 3.2, "dim_zona", ["id_zona · zona"], GRIS, w=1.5)
db = caja(9.1, 3.2, "dim_banda_icee", ["id_banda · banda", "límites 70 % y 95 %"], GRIS, w=1.7)
for (x1, y1, x2, y2) in [(hs["x"], hs["arriba"], dm["izq"] + 0.3, dm["abajo"]), (hc["x"], hc["arriba"], dm["der"] - 0.3, dm["abajo"]),
                         (hs["x"], hs["abajo"], dt_["izq"] + 0.3, dt_["arriba"]), (hc["x"], hc["abajo"], dt_["der"] - 0.3, dt_["arriba"]),
                         (dz["der"], dz["y"], hs["izq"], hs["y"]), (hc["der"], hc["y"], db["izq"], db["y"])]:
    ax.plot([x1, x2], [y1, y2], color="#52514e", lw=1, zorder=1)
ax.set_title("Modelo dimensional del proyecto: dos tablas de hechos que comparten las dimensiones municipio y tiempo", fontsize=10)
guardar_fig(fig, "A2_F01_esquema_estrella"); plt.show()

### 2.3 KPIs con fórmula, calculados con SQL sobre el modelo dimensional
Son los seis KPIs del protocolo de evaluación (Tabla 4 del documento). Cada uno se calcula con una consulta sobre el modelo y se compara con su línea base.

In [ ]:
kpi1 = sql("SELECT anio, AVG(punt_global) AS v FROM hechos_saber11 WHERE id_zona = 1 GROUP BY anio").set_index("anio")["v"]
kpi_urb = sql("SELECT anio, AVG(punt_global) AS v FROM hechos_saber11 WHERE id_zona = 0 GROUP BY anio").set_index("anio")["v"]
kpi2 = kpi_urb - kpi1
kpi3 = sql("""SELECT c.anio, 100.0 * SUM(c.vcs_rural) / SUM(c.vt_rural) AS v FROM hechos_cobertura c
              JOIN dim_tiempo t ON t.anio = c.anio WHERE t.tiene_saber11 = 1 GROUP BY c.anio""").set_index("anio")["v"]
vista = sql("SELECT * FROM vista_panel_rural")
kpi4 = vista.groupby(["anio", "id_banda"])["punt_global"].mean().unstack()
kpi4 = kpi4[3] - kpi4[1]
kpi4_total = vista[vista.id_banda == 3]["punt_global"].mean() - vista[vista.id_banda == 1]["punt_global"].mean()
kpi5 = vista.groupby("anio").apply(lambda g: stats.spearmanr(g["icee_rural"], g["punt_global"]).statistic)
kpi5_total = stats.spearmanr(vista["icee_rural"], vista["punt_global"]).statistic

# KPI 6: ΔR² del ICEE sobre los controles (los controles salen del panel del ACA 1, unido por llave)
controles = ["pct_estrato_0a2", "pct_madre_media", "pct_oficial", "pct_viviendas_rurales"]
base6 = vista.merge(panel[["cod_mpio", "anio"] + controles].rename(columns={"cod_mpio": "id_municipio"}), on=["id_municipio", "anio"])
anio_d = pd.get_dummies(base6["anio"], prefix="anio", drop_first=True, dtype=float)
def r2(cols):
    X = pd.concat([base6[cols], anio_d], axis=1)
    return LinearRegression().fit(X, base6["punt_global"]).score(X, base6["punt_global"])
kpi6_total = r2(controles + ["icee_rural"]) - r2(controles)
def r2_anio(g, cols):
    return LinearRegression().fit(g[cols], g["punt_global"]).score(g[cols], g["punt_global"])
kpi6 = base6.groupby("anio").apply(lambda g: r2_anio(g, controles + ["icee_rural"]) - r2_anio(g, controles))

KPI = pd.DataFrame([
    ["KPI 1. Puntaje global rural", "Promedio de punt_global de los estudiantes con id_zona = 1", "hechos_saber11", kpi1, None, "Línea base: 2019"],
    ["KPI 2. Brecha urbano–rural", "Promedio urbano − promedio rural de punt_global", "hechos_saber11 + dim_zona", kpi2, None, "Línea base: 2019"],
    ["KPI 3. ICEE rural nacional", "100 × Σ viviendas rurales con servicio / Σ viviendas rurales", "hechos_cobertura", kpi3, None, "Línea base: 2019"],
    ["KPI 4. Diferencia por cobertura", "Puntaje medio municipal banda Alta − banda Baja", "vista_panel_rural + dim_banda_icee", kpi4, kpi4_total, "Línea base: 0 puntos"],
    ["KPI 5. Asociación", "ρ de Spearman entre icee_rural y punt_global municipal", "vista_panel_rural", kpi5, kpi5_total, "Línea base: ρ = 0"],
    ["KPI 6. Varianza asociada", "R²(controles + ICEE + año) − R²(controles + año)", "vista_panel_rural + panel ACA 1", kpi6, kpi6_total, "Línea base: ΔR² = 0"],
], columns=["KPI", "Fórmula", "Tablas del modelo", "serie", "total", "Referencia"])
kpi_tab = KPI[["KPI", "Fórmula", "Tablas del modelo"]].copy()
for a_ in (2019, 2020, 2022):
    kpi_tab[str(a_)] = [s[a_] for s in KPI["serie"]]
kpi_tab["2019–2022"] = KPI["total"]
kpi_tab["Referencia"] = KPI["Referencia"]
kpi_tab = kpi_tab.round(4)
guardar_tabla(kpi_tab, "A2_T02_kpis")
registrar("kpis", kpi_tab.to_dict(orient="records"))
display(kpi_tab)

### 2.4 Validación de totales contra la fuente
Para cada tabla de hechos y cada KPI principal se compara el número de registros y una suma de control en la **fuente original** (los archivos crudos,
aplicando de forma independiente las reglas documentadas) y en el **modelo dimensional**. La diferencia esperada es cero; cuando no lo es, se explica.

In [ ]:
val = []
def control(objeto, medida, fuente, modelo, explicacion="Coinciden"):
    val.append({"Objeto": objeto, "Control": medida, "Fuente original": fuente, "Modelo o tablero": modelo,
                "Diferencia": modelo - fuente, "Explicación": explicacion})

# Recuento independiente sobre la fuente: una sola máscara con las reglas, sin pasar por las funciones del ETL
fuente = icfes_raw.drop_duplicates().drop_duplicates(subset=["estu_consecutivo"])
mascara = ((fuente["estu_estadoinvestigacion"] == "PUBLICAR") & (fuente["cole_calendario"] == "A")
           & ~fuente["cole_jornada"].isin(["SABATINA", "NOCHE"]) & fuente["cole_area_ubicacion"].isin(["RURAL", "URBANO"])
           & fuente["cole_cod_mcpio_ubicacion"].notna() & pd.to_numeric(fuente["punt_global"], errors="coerce").notna())
fuente = fuente[mascara]
mod_s = sql("SELECT anio, COUNT(*) AS n, SUM(punt_global) AS s FROM hechos_saber11 GROUP BY anio").set_index("anio")
crudo_n = icfes_raw.groupby("anio").size()
for a_ in (2019, 2020, 2022):
    control(f"hechos_saber11 {a_}", "Registros (crudos frente al modelo)", int(crudo_n[a_]), int(mod_s.loc[a_, "n"]),
            "Diferencia esperada: duplicados exactos y filtros del ETL (Tabla 7)")
    control(f"hechos_saber11 {a_}", "Registros (fuente con las reglas documentadas)", int((fuente["anio"] == a_).sum()), int(mod_s.loc[a_, "n"]))
    control(f"hechos_saber11 {a_}", "Suma de punt_global", float(pd.to_numeric(fuente.loc[fuente["anio"] == a_, "punt_global"]).sum()),
            float(mod_s.loc[a_, "s"]))
mod_c = sql("SELECT COUNT(*) AS n, SUM(vt_rural) AS vt FROM hechos_cobertura").iloc[0]
crudo_cob = pd.concat([icee_1922[["Viviendas totales - VT Rurales"]].rename(columns={"Viviendas totales - VT Rurales": "vt"}),
                       pd.concat([icee_23, icee_24])[["VT Rural"]].rename(columns={"VT Rural": "vt"})])
control("hechos_cobertura", "Filas (municipio–año)", len(crudo_cob), int(mod_c["n"]))
control("hechos_cobertura", "Suma de viviendas rurales (VT)", float(pd.to_numeric(crudo_cob["vt"]).sum()), float(mod_c["vt"]))
panel_aca1 = panel.rename(columns={"cod_mpio": "id_municipio"})
control("vista_panel_rural", "Filas municipio–año", len(panel_aca1), len(vista))
control("vista_panel_rural", "Estudiantes rurales", int(panel_aca1["n_estudiantes"].sum()), int(vista["n_estudiantes"].sum()))
control("vista_panel_rural", "Suma de puntajes municipales", float(panel_aca1["punt_global"].sum()), float(vista["punt_global"].sum()))
t10 = pd.read_csv(TAB / "T10_tendencia_nacional_puntajes.csv", encoding="utf-8-sig").set_index("anio")
t11 = pd.read_csv(TAB / "T11_tendencia_icee_rural.csv", encoding="utf-8-sig").set_index("anio")
for a_ in (2019, 2020, 2022):
    control(f"KPI 1 {a_}", "Puntaje rural (ACA 1 frente a SQL)", float(t10.loc[a_, "RURAL"]), round(float(kpi1[a_]), 2))
    control(f"KPI 2 {a_}", "Brecha urbano–rural (ACA 1 frente a SQL)", float(t10.loc[a_, "brecha"]), round(float(kpi2[a_]), 2))
    control(f"KPI 3 {a_}", "ICEE rural nacional (ACA 1 frente a SQL)", float(t11.loc[a_, "icee_rural_nacional"]), round(float(kpi3[a_]), 2))
control("KPI 4", "Diferencia Alta − Baja (ACA 1 frente a SQL)", float(cifras_aca1["diferencia_alta_baja"]), round(kpi4_total, 2))
control("KPI 5", "ρ de Spearman (ACA 1 frente a SQL)", round(rho_panel, 3), round(kpi5_total, 3))
val_tab = pd.DataFrame(val)
val_tab["Diferencia"] = val_tab["Diferencia"].round(6)
guardar_tabla(val_tab, "A2_T03_validacion_totales")
sin_explicar = val_tab[(val_tab["Diferencia"].abs() > 1e-6) & (val_tab["Explicación"] == "Coinciden")]
assert sin_explicar.empty, sin_explicar
registrar("validacion_controles", len(val_tab))
registrar("validacion_diferencias_cero", int((val_tab["Diferencia"].abs() <= 1e-6).sum()))
registrar("validacion_diferencias_explicadas", int((val_tab["Diferencia"].abs() > 1e-6).sum()))
display(val_tab)

## 3. Etapa 2 · Ingeniería de características y selección de variables
Se parte de las variables del panel del ACA 1 y se crean variables nuevas con significado de negocio. Reglas para evitar **fuga de información**:
(1) ninguna variable usa el puntaje rural (la variable objetivo) ni algo calculado con él (por eso se excluyen el puntaje urbano, la brecha
urbano–rural, la desviación estándar del puntaje, las áreas y la marca de atípico); (2) las variables de tiempo solo usan información del mismo año o de
años anteriores; (3) nada que aprenda de los datos (imputación, escalado, selección) se ajusta aquí: va dentro del `Pipeline` o se ajusta solo con el
conjunto de entrenamiento.

**Rezagos y ventanas.** El ICEE se publica para 2019–2024, así que para 2020 y 2022 existe el ICEE del año anterior. El de 2018 no está publicado a
escala municipal; por eso la ventana móvil usa los años disponibles (en 2019, solo 2019) y la variación se mide frente a 2019. No se usa el puntaje del año
anterior como variable: convertiría el modelo en autorregresivo y taparía el papel del contexto, que es lo que pregunta el objetivo 3.

In [ ]:
# ICEE de todos los años (2019–2024) para construir rezagos y ventanas sin mirar el futuro
icee_hist = sql("SELECT id_municipio, anio, icee_rural FROM hechos_cobertura").pivot(index="id_municipio", columns="anio", values="icee_rural")
def icee_ventana(fila):
    previo = icee_hist.loc[fila["cod_mpio"], fila["anio"] - 1] if fila["anio"] - 1 in icee_hist.columns else np.nan
    return np.nanmean([fila["icee_rural"], previo])

d = panel.copy()
d["icee_prom_movil_2a"] = d.apply(icee_ventana, axis=1)
d["variacion_icee_desde_2019"] = d["icee_rural"] - d["cod_mpio"].map(icee_hist[2019])
d["grupo_icee_ord"] = id_banda(d["icee_rural"]) - 1                       # 0 = Baja, 1 = Media, 2 = Alta (umbrales fijos)
d["conectividad_efectiva"] = d["icee_rural"] / 100 * d["pct_internet"]   # hogares con luz e internet (aprox.)
d["log_n_estudiantes"] = np.log(d["n_estudiantes"])
d["estudiantes_por_sede"] = d["n_estudiantes"] / d["n_sedes"]
for r_ in ["Amazonía", "Caribe", "Orinoquía", "Pacífica"]:                 # Andina = referencia
    d[f"region_{r_}"] = (d["region"] == r_).astype(int)

# Variables originales (medidas agregadas tal como salen de las fuentes) y variables creadas por ingeniería de características
BASE = ["icee_rural", "icee_urbano", "vt_rural", "pct_estrato_0a2", "pct_madre_media", "pct_internet", "pct_computador", "pct_oficial",
        "pct_mujer", "n_estudiantes", "n_sedes"]
CREADAS = ["brecha_icee_urb_rur", "log_vss_rural", "pct_viviendas_rurales", "grupo_icee_ord", "icee_prom_movil_2a", "variacion_icee_desde_2019",
           "conectividad_efectiva", "log_n_estudiantes", "estudiantes_por_sede", "region_Amazonía", "region_Caribe", "region_Orinoquía",
           "region_Pacífica", "pandemia", "post_pandemia"]
GRUPO = {**{v: "Energía" for v in ["icee_rural", "icee_urbano", "brecha_icee_urb_rur", "log_vss_rural", "grupo_icee_ord",
                                    "icee_prom_movil_2a", "variacion_icee_desde_2019", "conectividad_efectiva"]},
         **{v: "Socioeconómico" for v in ["pct_estrato_0a2", "pct_madre_media", "pct_internet", "pct_computador", "pct_oficial", "pct_mujer"]},
         **{v: "Territorio" for v in ["vt_rural", "pct_viviendas_rurales", "region_Amazonía", "region_Caribe", "region_Orinoquía", "region_Pacífica"]},
         **{v: "Escuela" for v in ["n_estudiantes", "n_sedes", "log_n_estudiantes", "estudiantes_por_sede"]},
         **{v: "Tiempo" for v in ["pandemia", "post_pandemia"]}}
CANDIDATAS = BASE + CREADAS
OBJETIVO = "punt_global"

FORMULA = {
    "brecha_icee_urb_rur": ("ICEE urbano − ICEE rural", "Desigualdad de cobertura dentro del municipio (ACA 1)"),
    "log_vss_rural": ("ln(1 + viviendas rurales sin servicio)", "Magnitud absoluta del déficit; el logaritmo reduce la asimetría (ACA 1)"),
    "pct_viviendas_rurales": ("100 × VT rural / VT total", "Grado de ruralidad del municipio (ACA 1)"),
    "grupo_icee_ord": ("0 si ICEE < 70; 1 si 70 ≤ ICEE < 95; 2 si ICEE ≥ 95", "Banda de cobertura del protocolo, codificada como ordinal"),
    "icee_prom_movil_2a": ("Promedio del ICEE rural de t y t − 1 (en 2019 solo t)", "Ventana móvil: la cobertura acumulada pesa más que la de un solo año"),
    "variacion_icee_desde_2019": ("ICEE rural(t) − ICEE rural(2019)", "Rezago: cambio de la cobertura frente al año base, sin mirar el futuro"),
    "conectividad_efectiva": ("ICEE rural / 100 × % con internet", "La luz habilita la conectividad (V de Cramér 0,196 en el ACA 1)"),
    "log_n_estudiantes": ("ln(estudiantes rurales evaluados)", "Tamaño del municipio rural; reduce la asimetría del conteo"),
    "estudiantes_por_sede": ("Estudiantes rurales / sedes rurales", "Tamaño medio de las sedes: escuelas dispersas frente a concentradas"),
    "region_Amazonía": ("1 si la región es Amazonía (one-hot; Andina = referencia)", "Codificación de la categórica región (heterogeneidad del ACA 1)"),
    "region_Caribe": ("1 si la región es Caribe", "Codificación de la categórica región"),
    "region_Orinoquía": ("1 si la región es Orinoquía", "Codificación de la categórica región"),
    "region_Pacífica": ("1 si la región es Pacífica", "Codificación de la categórica región"),
    "pandemia": ("1 si el año es 2020", "Variable de calendario: año de pandemia (referencia 2019)"),
    "post_pandemia": ("1 si el año es 2022", "Variable de calendario: año pospandemia (referencia 2019)"),
}
for v in BASE:
    FORMULA[v] = ("Medida original de la fuente, agregada por municipio y año", "Variable de dominio del ACA 1")
tabla_var = pd.DataFrame([{"Variable": v, "Origen": "Original" if v in BASE else "Creada", "Grupo": GRUPO[v],
                           "Fórmula": FORMULA[v][0], "Justificación": FORMULA[v][1],
                           "% faltantes": round(100 * d[v].isna().mean(), 2)} for v in CANDIDATAS])
print("Faltantes por variable candidata:", d[CANDIDATAS].isna().sum()[lambda s: s > 0].to_dict() or "ninguno")

### 3.1 Partición: el conjunto de prueba se separa antes de cualquier transformación
La unidad es el municipio–año y un mismo municipio aparece hasta tres veces. Si un municipio quedara a la vez en entrenamiento y en prueba, el modelo
«reconocería» al municipio y la evaluación sería optimista (fuga por grupo). Por eso la partición es **agrupada por municipio** (todos los años de un
municipio van al mismo lado) y **estratificada por banda de cobertura** (cada parte conserva la proporción de bandas Baja, Media y Alta).
El 20 % de los municipios queda como **prueba** y no se toca hasta el final.

In [ ]:
y_banda = d["grupo_icee_ord"].astype(int)
grupos = d["cod_mpio"]
corte = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
idx_ent, idx_prueba = next(corte.split(d, y_banda, grupos))
ent, prueba = d.iloc[idx_ent].reset_index(drop=True), d.iloc[idx_prueba].reset_index(drop=True)
assert set(ent["cod_mpio"]).isdisjoint(prueba["cod_mpio"])
part = pd.DataFrame({"Conjunto": ["Entrenamiento", "Prueba"], "Filas municipio–año": [len(ent), len(prueba)],
                     "Municipios": [ent["cod_mpio"].nunique(), prueba["cod_mpio"].nunique()],
                     "% banda Baja": [round(100 * (x["grupo_icee_ord"] == 0).mean(), 1) for x in (ent, prueba)],
                     "% banda Media": [round(100 * (x["grupo_icee_ord"] == 1).mean(), 1) for x in (ent, prueba)],
                     "% banda Alta": [round(100 * (x["grupo_icee_ord"] == 2).mean(), 1) for x in (ent, prueba)],
                     "Puntaje medio": [round(x[OBJETIVO].mean(), 2) for x in (ent, prueba)]})
guardar_tabla(part, "A2_T05_particion")
registrar("particion", part.to_dict(orient="records"))
display(part)

# Validación cruzada repetida sobre el entrenamiento: 5 pliegues agrupados y estratificados × 3 repeticiones = 15 particiones
REPETICIONES = [SEED, SEED + 1, SEED + 2]
def particiones(datos, semillas=REPETICIONES, k=5):
    lista = []
    for r_, s_ in enumerate(semillas, 1):
        cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=s_)
        for f_, (a_, b_) in enumerate(cv.split(datos, datos["grupo_icee_ord"].astype(int), datos["cod_mpio"]), 1):
            lista.append((r_, f_, a_, b_))
    return lista
CV = particiones(ent)
registrar("cv_particiones", len(CV))
print(f"Validación cruzada: {len(CV)} particiones (5 pliegues × {len(REPETICIONES)} repeticiones), agrupadas por municipio")

### 3.2 Selección de variables (solo con el conjunto de entrenamiento)
Se usan dos métodos que se complementan: **información mutua** (mide cuánta información aporta cada variable sobre el puntaje, también si la relación no es
lineal) y **LASSO** (una regresión que lleva a cero los coeficientes de las variables que sobran). LASSO se ajusta dentro de cada una de las 15 particiones de
entrenamiento, con imputación KNN y escalado en un `Pipeline`, y se cuenta en cuántas conserva cada variable (**estabilidad**). Quedan las variables que
LASSO conserva en al menos el 80 % de las particiones; el ICEE rural se mantiene siempre porque es la variable de interés del estudio.

In [ ]:
def pipe(modelo, imputador="knn"):
    imp = KNNImputer(n_neighbors=5) if imputador == "knn" else SimpleImputer(strategy="median")
    return Pipeline([("imputacion", imp), ("escalado", StandardScaler()), ("modelo", modelo)])

veces = pd.Series(0, index=CANDIDATAS)
for _, _, a_, _ in CV:
    p_ = pipe(LassoCV(cv=3, alphas=np.logspace(-3, 1, 40), random_state=SEED, max_iter=20000)).fit(ent.loc[a_, CANDIDATAS], ent.loc[a_, OBJETIVO])
    veces += (np.abs(p_.named_steps["modelo"].coef_) > 1e-8).astype(int)
estabilidad = veces / len(CV)
pre = Pipeline([("imputacion", KNNImputer(n_neighbors=5)), ("escalado", StandardScaler())]).fit(ent[CANDIDATAS])
im = mutual_info_regression(pre.transform(ent[CANDIDATAS]), ent[OBJETIVO], random_state=SEED)
seleccion = pd.DataFrame({"Variable": CANDIDATAS, "Información mutua": im, "Estabilidad LASSO": estabilidad.values})
seleccion["Rango IM"] = seleccion["Información mutua"].rank(ascending=False).astype(int)
seleccion["Queda"] = np.where((seleccion["Estabilidad LASSO"] >= 0.8) | (seleccion["Variable"] == "icee_rural"), "Sí", "No")
SELECCION = seleccion.loc[seleccion["Queda"] == "Sí", "Variable"].tolist()
tabla_var = tabla_var.merge(seleccion, on="Variable")
tabla_var = tabla_var.round({"Información mutua": 3, "Estabilidad LASSO": 2})
guardar_tabla(tabla_var, "A2_T04_variables_y_seleccion")
registrar("variables_candidatas", len(CANDIDATAS))
registrar("variables_creadas", len(CREADAS))
registrar("variables_seleccionadas", SELECCION)
registrar("variables_descartadas", [v for v in CANDIDATAS if v not in SELECCION])
CREADAS_SEL = [v for v in SELECCION if v in CREADAS]
BASE_SEL = [v for v in SELECCION if v in BASE]
print(f"Quedan {len(SELECCION)} de {len(CANDIDATAS)} variables: {SELECCION}")
display(tabla_var[["Variable", "Origen", "Grupo", "Información mutua", "Estabilidad LASSO", "Queda"]].sort_values("Información mutua", ascending=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5.2), sharey=True)
orden = seleccion.sort_values("Información mutua")
colores = [C1 if q == "Sí" else "#c9c8c3" for q in orden["Queda"]]
ax[0].barh(orden["Variable"], orden["Información mutua"], color=colores)
ax[0].set(title="Información mutua con el puntaje rural", xlabel="Información mutua (nats)")
ax[1].barh(orden["Variable"], orden["Estabilidad LASSO"], color=colores)
ax[1].axvline(0.8, color=C2, lw=1.2, ls="--"); ax[1].text(0.81, 0.3, "umbral 80 %", color=C2, fontsize=8)
ax[1].set(title="Estabilidad de LASSO en 15 particiones", xlabel="Proporción de particiones en que se conserva", xlim=(0, 1.05))
fig.suptitle("Selección de variables con el conjunto de entrenamiento (en azul, las que quedan)", fontsize=11, fontweight="bold")
fig.tight_layout(); guardar_fig(fig, "A2_F02_seleccion_variables"); plt.show()

## 4. Objetivo específico 2 · Pruebas de hipótesis H1 a H3
Las pruebas usan **todo el panel** (2.627 municipio–año): son inferencia sobre la población de municipios, no evaluación de un modelo, así que no
necesitan partición. Son **no paramétricas**, porque la normalidad se rechazó en el ACA 1 y el 37 % de los municipios tiene ICEE rural = 100 %.
Dentro de cada familia de hipótesis los valores p se corrigen con el método de **Holm** (controla la probabilidad de un falso positivo cuando se hacen
varias pruebas a la vez) y se decide con α = 0,05. Ninguna de las variables de estas pruebas tiene faltantes, así que no hay imputación.

| Hipótesis | Prueba | Tamaño del efecto |
|---|---|---|
| H1: ρ ≠ 0 entre ICEE rural y puntaje, por año y por región | Spearman + IC 95 % por bootstrap (2.000 remuestreos) | ρ |
| H2: el puntaje difiere entre las bandas Baja, Media y Alta | Kruskal–Wallis + comparaciones de Dunn | épsilon² |
| H3: el puntaje de la banda Alta supera al de la banda Baja | Mann–Whitney (una cola) | delta de Cliff |

In [ ]:
def holm(p):
    """Valores p ajustados por Holm (1979)."""
    p = np.asarray(p, float); orden_ = np.argsort(p); m = len(p)
    aj = np.empty(m); acumulado = 0
    for k, i in enumerate(orden_):
        acumulado = max(acumulado, min(1, (m - k) * p[i])); aj[i] = acumulado
    return aj

def ic_spearman(x, y, B=2000, rng=None):
    rng = rng or np.random.default_rng(SEED)
    x, y = np.asarray(x), np.asarray(y); n = len(x)
    vals = [stats.spearmanr(x[i], y[i]).statistic for i in (rng.integers(0, n, n) for _ in range(B))]
    return np.percentile(vals, [2.5, 97.5])

def cliff(a, b):
    """Delta de Cliff: P(a > b) − P(a < b), a partir de la U de Mann–Whitney."""
    u = stats.mannwhitneyu(a, b).statistic
    return 2 * u / (len(a) * len(b)) - 1

# H1 — asociación por año y por región
filas = []
rng_h1 = np.random.default_rng(SEED)
for tipo, col in [("Año", "anio"), ("Región", "region")]:
    for k, g in d.groupby(col):
        r_ = stats.spearmanr(g["icee_rural"], g[OBJETIVO])
        lo, hi = ic_spearman(g["icee_rural"], g[OBJETIVO], rng=rng_h1)
        filas.append({"Hipótesis": "H1", "Grupo": f"{tipo}: {k}", "n": len(g), "ρ de Spearman": r_.statistic, "IC 95 % inf.": lo,
                      "IC 95 % sup.": hi, "p": r_.pvalue})
h1 = pd.DataFrame(filas)
h1["p ajustado (Holm)"] = holm(h1["p"])
h1["Decisión"] = np.where(h1["p ajustado (Holm)"] < 0.05, "Se rechaza H0", "No se rechaza H0")
h1["Relevante (|ρ| ≥ 0,10 e IC sin 0)"] = np.where((h1["ρ de Spearman"].abs() >= 0.10) & ((h1["IC 95 % inf."] > 0) | (h1["IC 95 % sup."] < 0)), "Sí", "No")

# H2 — diferencias entre bandas (Kruskal–Wallis por año) y comparaciones por pares (Dunn)
def dunn(g, col_grupo, col_valor):
    """Prueba de Dunn con rangos conjuntos y corrección por empates; p sin ajustar."""
    rangos = stats.rankdata(g[col_valor]); n = len(g)
    _, conteo_empates = np.unique(g[col_valor], return_counts=True)
    corr = np.sum(conteo_empates ** 3 - conteo_empates) / (12 * (n - 1))
    res = []
    niveles = sorted(g[col_grupo].unique())
    for i in range(len(niveles)):
        for j in range(i + 1, len(niveles)):
            a_, b_ = g[col_grupo] == niveles[i], g[col_grupo] == niveles[j]
            z = (rangos[a_].mean() - rangos[b_].mean()) / np.sqrt((n * (n + 1) / 12 - corr) * (1 / a_.sum() + 1 / b_.sum()))
            res.append((niveles[i], niveles[j], z, 2 * stats.norm.sf(abs(z))))
    return res
NOMBRE_BANDA = {0: "Baja", 1: "Media", 2: "Alta"}
filas, filas_dunn = [], []
for a_, g in d.groupby("anio"):
    muestras = [g.loc[g["grupo_icee_ord"] == b, OBJETIVO] for b in (0, 1, 2)]
    kw = stats.kruskal(*muestras)
    eps2 = (kw.statistic - 3 + 1) / (len(g) - 3)           # épsilon² (Tomczak y Tomczak, 2014)
    filas.append({"Hipótesis": "H2", "Grupo": f"Año: {a_}", "n": len(g), "H de Kruskal–Wallis": kw.statistic, "épsilon²": eps2, "p": kw.pvalue,
                  "Mediana Baja": muestras[0].median(), "Mediana Media": muestras[1].median(), "Mediana Alta": muestras[2].median()})
    for b1, b2, z, p_ in dunn(g, "grupo_icee_ord", OBJETIVO):
        filas_dunn.append({"Año": a_, "Comparación": f"{NOMBRE_BANDA[b2]} frente a {NOMBRE_BANDA[b1]}", "z": z, "p": p_})
h2 = pd.DataFrame(filas)
h2["p ajustado (Holm)"] = holm(h2["p"])
h2["Decisión"] = np.where(h2["p ajustado (Holm)"] < 0.05, "Se rechaza H0", "No se rechaza H0")
h2_dunn = pd.DataFrame(filas_dunn)
h2_dunn["p ajustado (Holm)"] = holm(h2_dunn["p"])
h2_dunn["Decisión"] = np.where(h2_dunn["p ajustado (Holm)"] < 0.05, "Difieren", "No difieren")

# H3 — banda Alta frente a Baja (una cola: Alta > Baja), por año y por región
filas = []
for tipo, col in [("Año", "anio"), ("Región", "region")]:
    for k, g in d.groupby(col):
        alta, baja = g.loc[g["grupo_icee_ord"] == 2, OBJETIVO], g.loc[g["grupo_icee_ord"] == 0, OBJETIVO]
        if min(len(alta), len(baja)) < 5:
            filas.append({"Hipótesis": "H3", "Grupo": f"{tipo}: {k}", "n Alta": len(alta), "n Baja": len(baja), "Diferencia de medianas": np.nan,
                          "Delta de Cliff": np.nan, "p": np.nan})
            continue
        mw = stats.mannwhitneyu(alta, baja, alternative="greater")
        filas.append({"Hipótesis": "H3", "Grupo": f"{tipo}: {k}", "n Alta": len(alta), "n Baja": len(baja),
                      "Diferencia de medianas": alta.median() - baja.median(), "Delta de Cliff": cliff(alta, baja), "p": mw.pvalue})
h3 = pd.DataFrame(filas)
ok = h3["p"].notna()
h3.loc[ok, "p ajustado (Holm)"] = holm(h3.loc[ok, "p"])
h3["Decisión"] = np.where(~ok, "Sin datos suficientes (< 5 en una banda)",
                          np.where(h3["p ajustado (Holm)"] < 0.05, "Se rechaza H0", "No se rechaza H0"))
def magnitud_cliff(x):
    if pd.isna(x): return "—"
    x = abs(x)
    return "Despreciable" if x < 0.147 else ("Pequeña" if x < 0.33 else ("Mediana" if x < 0.474 else "Grande"))
h3["Magnitud (Romano et al., 2006)"] = h3["Delta de Cliff"].map(magnitud_cliff)

for t_, nombre in [(h1, "A2_T06_h1_spearman"), (h2, "A2_T07_h2_kruskal"), (h2_dunn, "A2_T08_h2_dunn"), (h3, "A2_T09_h3_mann_whitney")]:
    guardar_tabla(t_.round(4), nombre)
registrar("h1", h1.round(4).to_dict(orient="records"))
registrar("h2", h2.round(4).to_dict(orient="records"))
registrar("h2_dunn", h2_dunn.round(4).to_dict(orient="records"))
registrar("h3", h3.round(4).to_dict(orient="records"))

# Criterio de éxito del protocolo, parte 1: |ρ| ≥ 0,10 con IC sin cero en al menos 2 de 3 años
anios_ok = int(((h1["Grupo"].str.startswith("Año")) & (h1["Relevante (|ρ| ≥ 0,10 e IC sin 0)"] == "Sí")).sum())
registrar("criterio_exito_anios_relevantes", anios_ok)
print(f"Años con asociación relevante: {anios_ok} de 3 → criterio de éxito (parte 1) {'CUMPLIDO' if anios_ok >= 2 else 'NO cumplido'}")
display(h1.round(3)); display(h2.round(4)); display(h2_dunn.round(4)); display(h3.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw={"width_ratios": [1.1, 1]})
h1p = h1.iloc[::-1]
yy = np.arange(len(h1p))
colores = [C1 if g.startswith("Año") else C3 for g in h1p["Grupo"]]
ax[0].errorbar(h1p["ρ de Spearman"], yy, xerr=[h1p["ρ de Spearman"] - h1p["IC 95 % inf."], h1p["IC 95 % sup."] - h1p["ρ de Spearman"]],
               fmt="none", ecolor="#8a8984", elinewidth=1.2, capsize=3)
ax[0].scatter(h1p["ρ de Spearman"], yy, c=colores, s=40, zorder=3)
ax[0].axvline(0, color="#52514e", lw=0.8); ax[0].axvline(0.10, color=C2, lw=1, ls="--")
ax[0].text(0.105, len(h1p) - 0.6, "umbral 0,10", color=C2, fontsize=8)
ax[0].set_yticks(yy); ax[0].set_yticklabels([g.split(": ")[1] for g in h1p["Grupo"]])
ax[0].set(title="H1 · ρ de Spearman con IC 95 % (bootstrap)", xlabel="ρ entre ICEE rural y puntaje rural")
for k, (_, f_) in enumerate(h1p.iterrows()):
    ax[0].text(0.62, k, "✓" if f_["Decisión"] == "Se rechaza H0" else "×", fontsize=10, va="center",
               color=C3 if f_["Decisión"] == "Se rechaza H0" else C2)
ax[0].set_xlim(-0.2, 0.68)
h3p = h3[h3["p"].notna()].iloc[::-1]
yy = np.arange(len(h3p))
ax[1].barh(yy, h3p["Delta de Cliff"], color=[C1 if g.startswith("Año") else C3 for g in h3p["Grupo"]])
for lim, et in [(0.147, "pequeña"), (0.33, "mediana"), (0.474, "grande")]:
    ax[1].axvline(lim, color="#c9c8c3", lw=0.8, ls=":"); ax[1].text(lim + 0.005, len(h3p) - 0.5, et, fontsize=7, color="#52514e")
ax[1].set_yticks(yy); ax[1].set_yticklabels([g.split(": ")[1] for g in h3p["Grupo"]])
ax[1].axvline(0, color="#52514e", lw=0.8)
ax[1].set(title="H3 · delta de Cliff, banda Alta frente a Baja", xlabel="Delta de Cliff (−1 a 1)")
fig.suptitle("Pruebas del objetivo 2: azul = por año, verde = por región (✓ = significativa tras Holm)", fontsize=10.5, fontweight="bold")
fig.tight_layout(); guardar_fig(fig, "A2_F03_pruebas_h1_h3"); plt.show()

## 5. Objetivo específico 3 · Batería de modelos con validación repetida
El objetivo 3 pide **estimar cuánta variación del puntaje rural está asociada a la cobertura eléctrica**. La batería de modelos responde dos preguntas:
(1) ¿qué tan bien explican el puntaje las variables, medido **fuera de muestra** (en municipios que el modelo no vio)?, y (2) ¿cuánto empeora el mejor
modelo si se le quitan las variables de energía? La segunda es la respuesta del objetivo 3; la primera garantiza que no se interpreta un modelo que
memoriza los datos. No se busca predecir municipios nuevos: el modelo es una herramienta de diagnóstico.

**Diseño comparativo.** Todos los modelos usan las mismas variables (las seleccionadas), el mismo `Pipeline` (imputación KNN y escalado, ajustados solo con
la parte de entrenamiento de cada partición), las mismas 15 particiones, el mismo presupuesto de búsqueda de hiperparámetros (20 combinaciones al azar,
evaluadas con una validación aparte de 3 pliegues sobre el entrenamiento, nunca sobre la prueba) y las mismas métricas. La prueba se usa una sola vez, al final.

| Nivel | Modelos |
|---|---|
| (a) Línea base y clásico | Línea base ingenua (predice el promedio) · Regresión lineal |
| (b) Convencionales | Ridge · Lasso · KNN · SVR · Árbol de decisión |
| (c) Ensambles | Random Forest · HistGradientBoosting · XGBoost · LightGBM |
| (d) Red neuronal | Perceptrón multicapa (MLP) |

No se usan LSTM ni GRU: cada municipio tiene como máximo tres observaciones (2019, 2020 y 2022), una secuencia demasiado corta para una red recurrente.

In [ ]:
from scipy.stats import loguniform, randint, uniform
X_COLS = SELECCION
N_PRUEBAS = 20                      # mismo presupuesto de búsqueda para todos los modelos con hiperparámetros
registrar("n_pruebas_busqueda", N_PRUEBAS)

def con_escala_y(m):
    """Escala también la variable objetivo (necesario para SVR y la red neuronal)."""
    return lambda: TransformedTargetRegressor(regressor=m(), transformer=StandardScaler())

MODELOS = {   # nombre: (nivel, constructor, espacio de búsqueda, ¿escalar y?)
    "Línea base (promedio)": ("a", DummyRegressor, {}, False),
    "Regresión lineal": ("a", LinearRegression, {}, False),
    "Ridge": ("b", lambda: Ridge(random_state=SEED), {"alpha": loguniform(1e-3, 1e3)}, False),
    "Lasso": ("b", lambda: Lasso(random_state=SEED, max_iter=20000), {"alpha": loguniform(1e-4, 10)}, False),
    "KNN": ("b", KNeighborsRegressor, {"n_neighbors": randint(3, 60), "weights": ["uniform", "distance"], "p": [1, 2]}, False),
    "SVR": ("b", SVR, {"C": loguniform(0.05, 50), "epsilon": uniform(0.01, 0.5), "gamma": loguniform(1e-3, 1)}, True),
    "Árbol de decisión": ("b", lambda: DecisionTreeRegressor(random_state=SEED),
                          {"max_depth": randint(2, 13), "min_samples_leaf": randint(5, 61)}, False),
    "Random Forest": ("c", lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
                      {"n_estimators": randint(150, 501), "max_depth": [None, 6, 8, 10, 14, 18], "min_samples_leaf": randint(1, 21),
                       "max_features": [0.3, 0.5, 0.7, 1.0, "sqrt"]}, False),
    "HistGradientBoosting": ("c", lambda: HistGradientBoostingRegressor(random_state=SEED),
                             {"learning_rate": loguniform(0.01, 0.2), "max_iter": randint(100, 601), "max_leaf_nodes": randint(8, 65),
                              "min_samples_leaf": randint(10, 61), "l2_regularization": uniform(0, 5)}, False),
}
if HAY_XGB:
    from xgboost import XGBRegressor
    MODELOS["XGBoost"] = ("c", lambda: XGBRegressor(random_state=SEED, n_jobs=2, verbosity=0),
                          {"n_estimators": randint(100, 601), "learning_rate": loguniform(0.01, 0.2), "max_depth": randint(2, 9),
                           "subsample": uniform(0.6, 0.4), "colsample_bytree": uniform(0.6, 0.4), "reg_lambda": loguniform(0.1, 10)}, False)
if HAY_LGBM:
    from lightgbm import LGBMRegressor
    MODELOS["LightGBM"] = ("c", lambda: LGBMRegressor(random_state=SEED, n_jobs=2, verbose=-1),
                           {"n_estimators": randint(100, 601), "learning_rate": loguniform(0.01, 0.2), "num_leaves": randint(8, 65),
                            "min_child_samples": randint(10, 61), "subsample": uniform(0.6, 0.4), "subsample_freq": [1],
                            "colsample_bytree": uniform(0.6, 0.4), "reg_lambda": loguniform(0.1, 10)}, False)
MODELOS["Red neuronal (MLP)"] = ("d", lambda: MLPRegressor(random_state=SEED, max_iter=2000, early_stopping=True),
                                 {"hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)], "alpha": loguniform(1e-5, 1e-1),
                                  "learning_rate_init": loguniform(1e-4, 1e-2)}, True)
if not (HAY_XGB and HAY_LGBM):
    print("Aviso: XGBoost o LightGBM no están instalados en este entorno; la batería sigue sin ellos (en Google Colab sí se incluyen).")

def construir(nombre, params, imputador="knn"):
    nivel, fabrica, _, escala_y = MODELOS[nombre]
    base = fabrica()
    base.set_params(**params)
    p_ = pipe(base, imputador)
    return TransformedTargetRegressor(regressor=p_, transformer=StandardScaler()) if escala_y else p_

def metricas(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    e = y - yhat
    return {"rmse": float(np.sqrt(np.mean(e ** 2))), "mae": float(np.mean(np.abs(e))),
            "mape": float(100 * np.mean(np.abs(e) / np.abs(y))), "r2": float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2))}

### 5.1 Optimización de hiperparámetros (búsqueda aleatoria con el mismo presupuesto)
Se usa búsqueda aleatoria (`ParameterSampler` de scikit-learn) en lugar de Optuna para que el cuaderno no dependa de librerías adicionales; la guía admite
«Optuna u otra búsqueda». Cada combinación se evalúa con 3 pliegues agrupados por municipio **distintos** de los 15 que se usan para comparar los modelos,
para no elegir y evaluar con los mismos datos.

In [ ]:
cv_busqueda = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=2026).split(ent, ent["grupo_icee_ord"].astype(int), ent["cod_mpio"]))
Xe, ye = ent[X_COLS], ent[OBJETIVO]
mejores, filas_busqueda = {}, []
for nombre, (nivel, _, espacio, _) in MODELOS.items():
    t_ = time.time()
    candidatos = list(ParameterSampler(espacio, n_iter=N_PRUEBAS, random_state=SEED)) if espacio else [{}]
    resultados = []
    for params in candidatos:
        errores = [metricas(ye.iloc[b], construir(nombre, params).fit(Xe.iloc[a], ye.iloc[a]).predict(Xe.iloc[b]))["rmse"]
                   for a, b in cv_busqueda]
        resultados.append((np.mean(errores), params))
    mejor_rmse, mejor_params = min(resultados, key=lambda r: r[0])
    mejores[nombre] = mejor_params
    def legible(v):
        return round(float(v), 4) if isinstance(v, (float, np.floating)) else (int(v) if isinstance(v, (np.integer,)) else v)
    filas_busqueda.append({"Modelo": nombre, "Nivel": nivel,
                           "Espacio de búsqueda": "; ".join(f"{k}: {getattr(v, 'dist', None).name + str(tuple(round(x, 4) for x in v.args)) if hasattr(v, 'dist') else v}"
                                                            for k, v in espacio.items()) or "Sin hiperparámetros",
                           "Combinaciones probadas": len(candidatos),
                           "Mejores valores": "; ".join(f"{k} = {legible(v)}" for k, v in mejor_params.items()) or "—",
                           "RMSE de búsqueda": mejor_rmse, "Tiempo de búsqueda (s)": time.time() - t_})
    print(f"{nombre:<24} RMSE búsqueda {mejor_rmse:6.2f}  ({time.time() - t_:5.1f} s)  {mejor_params}")
busqueda = pd.DataFrame(filas_busqueda).round(3)
guardar_tabla(busqueda, "A2_T10_hiperparametros")
registrar("hiperparametros", busqueda.to_dict(orient="records"))

### 5.2 Evaluación en las 15 particiones y en la prueba
Cada modelo, con sus mejores hiperparámetros, se entrena en la parte de entrenamiento de cada partición y se mide en la parte de validación. Se guardan
todas las mediciones en `results/tables/metricas_por_particion.csv` (una fila por modelo y partición): con ese archivo se harán en el ACA 3 la prueba
estadística entre modelos y los intervalos de confianza. Al final cada modelo se entrena con todo el entrenamiento y se mide **una sola vez** en la prueba.

In [ ]:
def evaluar(nombre, params, columnas=None, particiones_=None, imputador="knn", etiqueta=None, datos=None):
    columnas = columnas or X_COLS; particiones_ = particiones_ or CV; datos = ent if datos is None else datos
    filas_ = []
    for r_, f_, a_, b_ in particiones_:
        m_ = construir(nombre, params, imputador)
        t_ = time.time(); m_.fit(datos.loc[a_, columnas], datos.loc[a_, OBJETIVO]); tt = time.time() - t_
        filas_.append({"modelo": etiqueta or nombre, "nivel": MODELOS[nombre][0], "repeticion": r_, "pliegue": f_,
                       "n_entrenamiento": len(a_), "n_validacion": len(b_), "tiempo_entrenamiento_s": tt,
                       **metricas(datos.loc[b_, OBJETIVO], m_.predict(datos.loc[b_, columnas]))})
    return pd.DataFrame(filas_)

por_particion = pd.concat([evaluar(n, mejores[n]) for n in MODELOS], ignore_index=True)

# Prueba (una sola vez por modelo)
filas_prueba, PRED_PRUEBA, AJUSTADOS = [], {}, {}
for nombre in MODELOS:
    m_ = construir(nombre, mejores[nombre])
    t_ = time.time(); m_.fit(ent[X_COLS], ent[OBJETIVO]); tt = time.time() - t_
    PRED_PRUEBA[nombre] = m_.predict(prueba[X_COLS]); AJUSTADOS[nombre] = m_
    filas_prueba.append({"Modelo": nombre, "tiempo_total_s": tt, **{f"prueba_{k}": v for k, v in metricas(prueba[OBJETIVO], PRED_PRUEBA[nombre]).items()}})
prueba_tab = pd.DataFrame(filas_prueba)

def resumen(pp):
    g = pp.groupby("modelo", sort=False)
    r_ = pd.DataFrame({"Nivel": g["nivel"].first(),
                       "RMSE media": g["rmse"].mean(), "RMSE d.e.": g["rmse"].std(),
                       "MAE media": g["mae"].mean(), "MAE d.e.": g["mae"].std(),
                       "MAPE % media": g["mape"].mean(), "MAPE % d.e.": g["mape"].std(),
                       "R² media": g["r2"].mean(), "R² d.e.": g["r2"].std(),
                       "Tiempo medio de entrenamiento (s)": g["tiempo_entrenamiento_s"].mean()})
    return r_.reset_index().rename(columns={"modelo": "Modelo"})
comparacion = resumen(por_particion).merge(prueba_tab, on="Modelo")
comparacion = comparacion.sort_values("RMSE media").reset_index(drop=True)
MEJOR = comparacion.loc[comparacion["Modelo"] != "Línea base (promedio)", "Modelo"].iloc[0]
registrar("mejor_modelo", MEJOR)
guardar_tabla(comparacion.round(4), "A2_T11_comparacion_modelos")
registrar("comparacion_modelos", comparacion.round(4).to_dict(orient="records"))
display(comparacion.round(3))

In [ ]:
# Figura que compara todos los modelos: RMSE en las 15 particiones
orden_m = comparacion["Modelo"].tolist()
COLOR_NIVEL = {"a": GRIS, "b": C1, "c": C3, "d": C2}
fig, ax = plt.subplots(figsize=(10, 5))
datos_box = [por_particion.loc[por_particion["modelo"] == m, "rmse"] for m in orden_m]
bp = ax.boxplot(datos_box, orientation="horizontal", patch_artist=True, widths=0.6, medianprops={"color": "#1d2433"})
for caja_, m in zip(bp["boxes"], orden_m):
    caja_.set_facecolor(COLOR_NIVEL[MODELOS[m][0]]); caja_.set_alpha(0.75)
ax.set_yticks(range(1, len(orden_m) + 1)); ax.set_yticklabels(orden_m)
ax.invert_yaxis()
for k, m in enumerate(orden_m, 1):
    fila = comparacion[comparacion["Modelo"] == m].iloc[0]
    ax.text(ax.get_xlim()[1] if k == 1 else ax.get_xlim()[1], k, f" {fila['RMSE media']:.2f} ± {fila['RMSE d.e.']:.2f}", va="center", fontsize=8)
ax.set(title="RMSE de validación en 15 particiones agrupadas por municipio (menor es mejor)", xlabel="RMSE (puntos de puntaje global)")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COLOR_NIVEL[k], label=l) for k, l in [("a", "(a) Línea base y clásico"), ("b", "(b) Convencionales"),
                                                                      ("c", "(c) Ensambles"), ("d", "(d) Red neuronal")]],
          loc="lower right", fontsize=8)
fig.tight_layout(); guardar_fig(fig, "A2_F04_comparacion_modelos"); plt.show()

# Figura del mejor modelo: real contra predicho y residuos (conjunto de prueba)
yhat = PRED_PRUEBA[MEJOR]; yreal = prueba[OBJETIVO].to_numpy(); res = yreal - yhat
fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.3))
ax[0].scatter(yreal, yhat, s=10, alpha=0.5, color=C1)
lim = [min(yreal.min(), yhat.min()) - 5, max(yreal.max(), yhat.max()) + 5]
ax[0].plot(lim, lim, color=C2, lw=1.2, label="Predicción perfecta"); ax[0].legend(loc="upper left", fontsize=8)
ax[0].set(xlim=lim, ylim=lim, title=f"{MEJOR}: real contra predicho (prueba)", xlabel="Puntaje rural real", ylabel="Puntaje rural predicho")
ax[1].scatter(yhat, res, s=10, alpha=0.5, color=C1); ax[1].axhline(0, color=C2, lw=1.2)
ax[1].set(title="Residuos contra predicho (prueba)", xlabel="Puntaje rural predicho", ylabel="Residuo (real − predicho)")
fig.tight_layout(); guardar_fig(fig, "A2_F05_mejor_modelo"); plt.show()

### 5.3 Ablación: el mejor modelo con y sin la ingeniería de características
Se repite la evaluación del mejor modelo en las mismas 15 particiones con tres conjuntos de variables. Como las particiones son las mismas, las
diferencias se miden **por pares** (partición a partición).

In [ ]:
CONJUNTOS = [("Variables originales, sin selección", BASE), ("Variables originales seleccionadas", BASE_SEL),
             ("Originales + creadas, seleccionadas (final)", X_COLS)]
abl = pd.concat([evaluar(MEJOR, mejores[MEJOR], columnas=c, etiqueta=n) for n, c in CONJUNTOS], ignore_index=True)
ref = abl[abl["modelo"] == CONJUNTOS[-1][0]].set_index(["repeticion", "pliegue"])
filas = []
for n, c in CONJUNTOS:
    s_ = abl[abl["modelo"] == n].set_index(["repeticion", "pliegue"])
    filas.append({"Conjunto de variables": n, "Variables": len(c), "RMSE media": s_["rmse"].mean(), "RMSE d.e.": s_["rmse"].std(),
                  "R² media": s_["r2"].mean(), "R² d.e.": s_["r2"].std(),
                  "ΔRMSE frente al final (media)": (s_["rmse"] - ref["rmse"]).mean(), "ΔRMSE d.e.": (s_["rmse"] - ref["rmse"]).std()})
ablacion = pd.DataFrame(filas).round(4)
guardar_tabla(ablacion, "A2_T12_ablacion")
registrar("ablacion", ablacion.to_dict(orient="records"))
display(ablacion)

### 5.4 ¿Cuánto aporta cada grupo de variables? (idea «Suma contexto») — la respuesta del objetivo 3
Se quita **un grupo completo de variables a la vez** del mejor modelo y se mide cuánto empeora fuera de muestra. El aumento del RMSE (y la caída del R²) al
quitar el grupo **Energía** es la variación del puntaje que solo la cobertura eléctrica permite reproducir, después de tener en cuenta los demás grupos.

In [ ]:
filas = []
for grupo in ["Energía", "Socioeconómico", "Territorio", "Escuela", "Tiempo"]:
    cols = [c for c in X_COLS if GRUPO[c] != grupo]
    quitadas = [c for c in X_COLS if GRUPO[c] == grupo]
    if not quitadas:
        continue
    s_ = evaluar(MEJOR, mejores[MEJOR], columnas=cols, etiqueta=f"Sin {grupo}").set_index(["repeticion", "pliegue"])
    d_rmse, d_r2 = s_["rmse"] - ref["rmse"], ref["r2"] - s_["r2"]
    filas.append({"Grupo quitado": grupo, "Variables quitadas": ", ".join(quitadas), "RMSE sin el grupo": s_["rmse"].mean(),
                  "Aumento del RMSE (media)": d_rmse.mean(), "Aumento del RMSE (d.e.)": d_rmse.std(),
                  "Caída del R² (media)": d_r2.mean(), "Caída del R² (d.e.)": d_r2.std(),
                  "Particiones en que empeora": int((d_rmse > 0).sum())})
contexto = pd.DataFrame(filas).sort_values("Aumento del RMSE (media)", ascending=False).round(4)
guardar_tabla(contexto, "A2_T13_suma_contexto")
registrar("suma_contexto", contexto.to_dict(orient="records"))
display(contexto)

# ¿Con qué otra información se superpone la energía? Se mide cuánto mejora el modelo al agregar las variables de energía según qué otras variables
# ya tiene: solo los controles socioeconómicos estructurales, más el territorio (región y ruralidad), más la conectividad del hogar (internet y
# computador, que solo son posibles con luz: variables mediadoras), o todo. Se usa el mejor modelo y, como referencia interpretable, la regresión lineal.
MEDIADORAS = ["pct_internet", "pct_computador", "conectividad_efectiva"]
ENERGIA = [c for c in X_COLS if GRUPO[c] == "Energía" and c not in MEDIADORAS]
SOCIO = [c for c in X_COLS if GRUPO[c] in ("Socioeconómico", "Escuela", "Tiempo") and c not in MEDIADORAS]
TERR = [c for c in X_COLS if GRUPO[c] == "Territorio"]
CONEC = [c for c in X_COLS if c in MEDIADORAS]
NIVELES = [("Controles socioeconómicos", SOCIO), ("+ territorio (región y ruralidad)", SOCIO + TERR),
           ("+ conectividad del hogar", SOCIO + CONEC), ("+ territorio y conectividad (todo)", SOCIO + TERR + CONEC)]
filas = []
for modelo_ in [MEJOR, "Regresión lineal"]:
    for nombre_, cols_ in NIVELES:
        s0 = evaluar(modelo_, mejores[modelo_], columnas=cols_).set_index(["repeticion", "pliegue"])
        s1 = evaluar(modelo_, mejores[modelo_], columnas=cols_ + ENERGIA).set_index(["repeticion", "pliegue"])
        mejora, d_r2 = s0["rmse"] - s1["rmse"], s1["r2"] - s0["r2"]
        filas.append({"Modelo": modelo_, "Variables de control": nombre_, "RMSE sin energía": s0["rmse"].mean(), "RMSE con energía": s1["rmse"].mean(),
                      "Mejora del RMSE (media)": mejora.mean(), "Mejora del RMSE (d.e.)": mejora.std(), "ΔR² (media)": d_r2.mean(),
                      "Particiones en que mejora": int((mejora > 0).sum())})
aporte = pd.DataFrame(filas).round(4)
guardar_tabla(aporte, "A2_T24_aporte_energia")
registrar("aporte_energia", aporte.to_dict(orient="records"))
registrar("variables_energia", ENERGIA)
display(aporte)

# Importancia por permutación en la prueba (cuánto empeora el modelo final si se baraja cada variable)
pi = permutation_importance(AJUSTADOS[MEJOR], prueba[X_COLS], prueba[OBJETIVO], n_repeats=30, random_state=SEED,
                            scoring="neg_root_mean_squared_error")
imp = pd.DataFrame({"Variable": X_COLS, "Grupo": [GRUPO[c] for c in X_COLS], "Aumento del RMSE al permutar": pi.importances_mean,
                    "d.e.": pi.importances_std}).sort_values("Aumento del RMSE al permutar", ascending=False).round(4)
guardar_tabla(imp, "A2_T14_importancia_permutacion")
registrar("importancia_permutacion", imp.to_dict(orient="records"))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
cx = contexto.iloc[::-1]
ax[0].barh(cx["Grupo quitado"], cx["Aumento del RMSE (media)"], xerr=cx["Aumento del RMSE (d.e.)"],
           color=[C2 if g == "Energía" else C1 for g in cx["Grupo quitado"]], ecolor="#8a8984", capsize=3)
ax[0].axvline(0, color="#52514e", lw=0.8)
ax[0].set(title="Aumento del RMSE al quitar cada grupo", xlabel="Δ RMSE (puntos), media ± d.e. en 15 particiones")
im_ = imp.iloc[::-1]
ax[1].barh(im_["Variable"], im_["Aumento del RMSE al permutar"], xerr=im_["d.e."],
           color=[C2 if GRUPO[v] == "Energía" else C1 for v in im_["Variable"]], ecolor="#8a8984", capsize=2)
ax[1].set(title=f"Importancia por permutación ({MEJOR}, prueba)", xlabel="Aumento del RMSE al barajar la variable (puntos)")
fig.suptitle("¿Qué información explica el puntaje rural? En naranja, la energía", fontsize=10.5, fontweight="bold")
fig.tight_layout(); guardar_fig(fig, "A2_F06_contexto_e_importancia"); plt.show()

### 5.5 Comparación con alternativas más complejas y prueba de que el resultado se sostiene
**Compara alternativas:** se combinan los tres mejores modelos por apilamiento (*stacking*, con una regresión Ridge que aprende cómo pesarlos) y por votación
(promedio simple), en las mismas particiones. **Demuestra que se sostiene:** se repite toda la evaluación con otras tres semillas de partición y se mide si el
orden de los modelos se mantiene; además, una validación temporal: entrenar con 2019 y 2020 y medir en 2022 en los municipios de prueba.

In [ ]:
top3 = [m for m in comparacion["Modelo"] if m != "Línea base (promedio)"][:3]
registrar("top3", top3)
def ensamble(tipo):
    est_ = [(f"m{k}", construir(m, mejores[m])) for k, m in enumerate(top3)]
    return (StackingRegressor(est_, final_estimator=Ridge(alpha=1.0), cv=3, n_jobs=1) if tipo == "Stacking"
            else VotingRegressor(est_))
filas = []
for tipo in ["Stacking", "Votación"]:
    for r_, f_, a_, b_ in CV:
        m_ = ensamble(tipo); t_ = time.time(); m_.fit(ent.loc[a_, X_COLS], ent.loc[a_, OBJETIVO]); tt = time.time() - t_
        filas.append({"modelo": f"{tipo} ({', '.join(top3)})", "nivel": "c", "repeticion": r_, "pliegue": f_, "n_entrenamiento": len(a_),
                      "n_validacion": len(b_), "tiempo_entrenamiento_s": tt, **metricas(ent.loc[b_, OBJETIVO], m_.predict(ent.loc[b_, X_COLS]))})
ens_pp = pd.DataFrame(filas)
ens = resumen(ens_pp)
pr = []
for tipo, nombre in zip(["Stacking", "Votación"], ens["Modelo"]):
    m_ = ensamble(tipo).fit(ent[X_COLS], ent[OBJETIVO])
    pr.append({"Modelo": nombre, **{f"prueba_{k}": v for k, v in metricas(prueba[OBJETIVO], m_.predict(prueba[X_COLS])).items()}})
ens = ens.merge(pd.DataFrame(pr), on="Modelo")
mejor_fila = comparacion[comparacion["Modelo"] == MEJOR].copy()
alternativas = pd.concat([mejor_fila[ens.columns.intersection(mejor_fila.columns)], ens], ignore_index=True).round(4)
guardar_tabla(alternativas, "A2_T15_alternativas_ensamble")
registrar("alternativas", alternativas.to_dict(orient="records"))
display(alternativas)

# Otras tres semillas de partición: ¿se mantiene el orden de los modelos?
CV2 = particiones(ent, semillas=[SEED + 10, SEED + 11, SEED + 12])
otra = pd.concat([evaluar(n, mejores[n], particiones_=CV2) for n in MODELOS], ignore_index=True)
orden2 = otra.groupby("modelo")["rmse"].mean()
orden1 = por_particion.groupby("modelo")["rmse"].mean()
rho_orden = stats.spearmanr(orden1[orden2.index], orden2).statistic
estabilidad_orden = pd.DataFrame({"Modelo": orden1.index, "RMSE semillas 42–44": orden1.values, "Puesto 42–44": orden1.rank().astype(int).values,
                                  "RMSE semillas 52–54": orden2[orden1.index].values, "Puesto 52–54": orden2[orden1.index].rank().astype(int).values}
                                 ).sort_values("Puesto 42–44").round(4)
# Validación temporal: entrena 2019–2020 (municipios de entrenamiento) y mide en 2022 (municipios de prueba)
ent_t, val_t = ent[ent["anio"] < 2022], prueba[prueba["anio"] == 2022]
estabilidad_orden["RMSE temporal (2022)"] = [metricas(val_t[OBJETIVO], construir(m, mejores[m]).fit(ent_t[X_COLS], ent_t[OBJETIVO])
                                                     .predict(val_t[X_COLS]))["rmse"] for m in estabilidad_orden["Modelo"]]
estabilidad_orden["Puesto temporal"] = estabilidad_orden["RMSE temporal (2022)"].rank().astype(int)
estabilidad_orden = estabilidad_orden.round(4)
guardar_tabla(estabilidad_orden, "A2_T16_estabilidad_orden")
registrar("estabilidad_orden", estabilidad_orden.to_dict(orient="records"))
registrar("rho_orden_semillas", round(float(rho_orden), 3))
registrar("rho_orden_temporal", round(float(stats.spearmanr(estabilidad_orden["Puesto 42–44"], estabilidad_orden["Puesto temporal"]).statistic), 3))
print(f"Correlación de los puestos entre semillas: {rho_orden:.3f}")
display(estabilidad_orden)

### 5.6 Sensibilidad a la imputación (ajuste pedido en la retroalimentación del ACA 1)
Las variables del panel no tienen faltantes (los porcentajes municipales se calcularon sobre respuestas válidas), así que el imputador del `Pipeline` es una
salvaguarda para datos nuevos. Para medir si el método importa, en cada partición se ocultan al azar el 10 % de los valores de cada variable **solo en la parte
de entrenamiento**; el imputador (KNN o mediana) se ajusta con esa parte y el modelo se mide en la validación intacta. Así la imputación nunca ve los datos
de validación.

In [ ]:
rng_s = np.random.default_rng(SEED + 7)
filas = []
for imputador in ["knn", "mediana"]:
    for r_, f_, a_, b_ in CV:
        Xa = ent.loc[a_, X_COLS].copy()
        mascara_s = np.random.default_rng(1000 * r_ + f_).random(Xa.shape) < 0.10
        Xa = Xa.mask(mascara_s)
        m_ = construir(MEJOR, mejores[MEJOR], "knn" if imputador == "knn" else "median").fit(Xa, ent.loc[a_, OBJETIVO])
        filas.append({"Imputador": "KNN (5 vecinos)" if imputador == "knn" else "Mediana", "repeticion": r_, "pliegue": f_,
                      **metricas(ent.loc[b_, OBJETIVO], m_.predict(ent.loc[b_, X_COLS]))})
sens = pd.DataFrame(filas)
sens_res = sens.groupby("Imputador").agg(**{"RMSE media": ("rmse", "mean"), "RMSE d.e.": ("rmse", "std"), "R² media": ("r2", "mean"),
                                            "R² d.e.": ("r2", "std")}).reset_index()
dif = (sens[sens["Imputador"] == "Mediana"].set_index(["repeticion", "pliegue"])["rmse"]
       - sens[sens["Imputador"] == "KNN (5 vecinos)"].set_index(["repeticion", "pliegue"])["rmse"])
sens_res["Sin ocultar (referencia)"] = float(ref["rmse"].mean())
sens_res = sens_res.round(4)
guardar_tabla(sens_res, "A2_T17_sensibilidad_imputacion_modelo")
registrar("sensibilidad_imputacion_modelo", sens_res.to_dict(orient="records"))
registrar("sensibilidad_dif_mediana_menos_knn", {"media": round(float(dif.mean()), 4), "de": round(float(dif.std()), 4),
                                                 "particiones_knn_mejor": int((dif > 0).sum())})
display(sens_res)

# Métricas por partición (una fila por modelo y partición) para la prueba estadística del ACA 3
todo_pp = pd.concat([por_particion, ens_pp], ignore_index=True)
todo_pp["semillas_particion"] = "42-44"
todo_pp.round(6).to_csv(TAB / "metricas_por_particion.csv", index=False, encoding="utf-8")
registrar("metricas_por_particion_filas", len(todo_pp))

## 6. Objetivo específico 3 · Modelo de datos panel (H4 a H6)
La batería mide cuánto aporta la energía **fuera de muestra**; el modelo de panel da la **estimación con incertidumbre** que pide el protocolo:
cuántos puntos de puntaje se asocian a 10 puntos porcentuales más de cobertura rural, con su intervalo de confianza y su valor p.

* **Modelo A (agrupado):** puntaje = ICEE rural + controles + efectos fijos de año + efectos fijos de región. Compara municipios entre sí.
* **Modelo B (efectos fijos de municipio y de año):** resta a cada municipio su propio promedio, así que solo usa los **cambios dentro del mismo municipio**
  entre 2019, 2020 y 2022. Elimina todo lo que no cambia en el municipio (geografía, historia, cultura), pero necesita que la cobertura cambie con el tiempo.

Los controles son los del ACA 1 (estrato 0–2, madre con media o más, colegio oficial y grado de ruralidad). No se controla por internet ni computador:
la luz es condición para tenerlos, así que controlar por ellos restaría parte de la asociación que se quiere medir (serían variables mediadoras).
Los errores estándar son **agrupados por municipio** (las tres observaciones de un municipio no son independientes; Wooldridge, 2010). Como sensibilidad,
se agrupan por departamento, porque el I de Moran del ACA 1 mostró que los municipios de un mismo departamento se parecen.

In [ ]:
def es(x, d=2):
    """Número con coma decimal, como en el documento."""
    return f"{x:,.{d}f}".replace(",", "§").replace(".", ",").replace("§", ".")
def p_es(p):
    return "< 0,001" if p < 0.001 else es(p, 3)

CTRL = ["pct_estrato_0a2", "pct_madre_media", "pct_oficial", "pct_viviendas_rurales"]
REG = ["region_Amazonía", "region_Caribe", "region_Orinoquía", "region_Pacífica"]
dp = d.copy()
dp["icee10"] = dp["icee_rural"] / 10                       # coeficiente = puntos por cada 10 puntos porcentuales de ICEE
dp["anio_2020"], dp["anio_2022"] = dp["pandemia"], dp["post_pandemia"]

def ols_cluster(datos, y, xs, cluster, intercepto=True):
    """MCO con errores estándar robustos agrupados (CR1, como Stata). Devuelve coeficientes, matriz de covarianza y medidas de ajuste."""
    X = datos[xs].to_numpy(float)
    if intercepto:
        X = np.column_stack([np.ones(len(X)), X]); nombres = ["const"] + xs
    else:
        nombres = list(xs)
    yv = datos[y].to_numpy(float); n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    b = XtX_inv @ X.T @ yv
    e = yv - X @ b
    S = pd.DataFrame(X * e[:, None]).groupby(datos[cluster].to_numpy()).sum().to_numpy()
    G = S.shape[0]
    V = (G / (G - 1)) * ((n - 1) / (n - k)) * XtX_inv @ (S.T @ S) @ XtX_inv
    se = np.sqrt(np.diag(V))
    tcrit = stats.t.ppf(0.975, G - 1)
    tab = pd.DataFrame({"coef": b, "ee": se, "t": b / se, "p": 2 * stats.t.sf(np.abs(b / se), G - 1),
                        "ic_inf": b - tcrit * se, "ic_sup": b + tcrit * se}, index=nombres)
    r2 = 1 - np.sum(e ** 2) / np.sum((yv - yv.mean()) ** 2)
    return {"tabla": tab, "V": pd.DataFrame(V, index=nombres, columns=nombres), "n": n, "G": G, "r2": r2, "residuos": e}

def wald(res, terminos):
    b = res["tabla"].loc[terminos, "coef"].to_numpy(); V = res["V"].loc[terminos, terminos].to_numpy()
    q = len(terminos); F = float(b @ np.linalg.solve(V, b)) / q
    return F, q, float(stats.f.sf(F, q, res["G"] - 1))

def combinacion(res, pesos):
    """Coeficiente y error estándar de una suma de coeficientes (p. ej., pendiente de una región = ICEE + ICEE × región)."""
    w = pd.Series(pesos); b = float(res["tabla"].loc[w.index, "coef"] @ w)
    se = float(np.sqrt(w.to_numpy() @ res["V"].loc[w.index, w.index].to_numpy() @ w.to_numpy()))
    tcrit = stats.t.ppf(0.975, res["G"] - 1)
    return b, se, b - tcrit * se, b + tcrit * se, 2 * stats.t.sf(abs(b / se), res["G"] - 1)

def dentro(datos, columnas, grupo="cod_mpio"):
    """Transformación «within»: resta a cada variable el promedio de su municipio (solo municipios con 2 o más años)."""
    g = datos[datos.groupby(grupo)[grupo].transform("size") >= 2].copy()
    for c in columnas:
        g[c] = g[c] - g.groupby(grupo)[c].transform("mean")
    return g

# Modelo A y modelo B, con y sin interacciones
dp["icee10_x_2020"], dp["icee10_x_2022"] = dp["icee10"] * dp["anio_2020"], dp["icee10"] * dp["anio_2022"]
for r_ in REG:
    dp[f"icee10_x_{r_}"] = dp["icee10"] * dp[r_]
INT_ANIO = ["icee10_x_2020", "icee10_x_2022"]
INT_REG = [f"icee10_x_{r_}" for r_ in REG]
XA = ["icee10"] + CTRL + ["anio_2020", "anio_2022"] + REG
XB = ["icee10"] + CTRL + ["anio_2020", "anio_2022"]
A = ols_cluster(dp, OBJETIVO, XA, "cod_mpio")
A_dep = ols_cluster(dp, OBJETIVO, XA, "cod_depto")
A_sin = ols_cluster(dp, OBJETIVO, ["icee10"] + CTRL + ["anio_2020", "anio_2022"], "cod_mpio")    # el modelo del KPI 6 (sin región)
A_ctrl = ols_cluster(dp, OBJETIVO, CTRL + ["anio_2020", "anio_2022"], "cod_mpio")
A_reg = ols_cluster(dp, OBJETIVO, XA + INT_REG, "cod_mpio")
w_ = dentro(dp, [OBJETIVO] + XB + INT_ANIO + INT_REG)
B = ols_cluster(w_, OBJETIVO, XB, "cod_mpio", intercepto=False)
B_dep = ols_cluster(w_, OBJETIVO, XB, "cod_depto", intercepto=False)
B_anio = ols_cluster(w_, OBJETIVO, XB + INT_ANIO, "cod_mpio", intercepto=False)
B_reg = ols_cluster(w_, OBJETIVO, XB + INT_REG, "cod_mpio", intercepto=False)
registrar("panel_fe_obs", B["n"]); registrar("panel_fe_municipios", B["G"])
var_dentro = ((10 * w_["icee10"]) ** 2).mean() ** 0.5          # desviación del ICEE dentro de cada municipio (ya centrado)
var_total = dp["icee_rural"].std()
registrar("icee_de_dentro", round(float(var_dentro), 3)); registrar("icee_de_total", round(float(var_total), 3))
registrar("icee_pct_municipios_sin_cambio", round(100 * float((dp.groupby("cod_mpio")["icee_rural"].agg(lambda s: s.max() - s.min()) < 0.5).mean()), 1))

def fila_modelo(nombre, res, termino="icee10", agrupado="Municipio"):
    t_ = res["tabla"].loc[termino]
    return {"Modelo": nombre, "Errores agrupados por": agrupado, "Coef. ICEE (puntos por 10 p.p.)": t_["coef"], "Error estándar": t_["ee"],
            "IC 95 % inf.": t_["ic_inf"], "IC 95 % sup.": t_["ic_sup"], "p": t_["p"], "R²": res["r2"], "Observaciones": res["n"], "Grupos": res["G"]}
modelos_panel = pd.DataFrame([
    fila_modelo("A · Agrupado: año + controles (modelo del KPI 6)", A_sin),
    fila_modelo("A · Agrupado: año + región + controles", A),
    fila_modelo("A · Agrupado: año + región + controles", A_dep, agrupado="Departamento"),
    fila_modelo("B · Efectos fijos de municipio y año + controles", B),
    fila_modelo("B · Efectos fijos de municipio y año + controles", B_dep, agrupado="Departamento"),
]).round(4)
guardar_tabla(modelos_panel, "A2_T18_modelos_panel")
registrar("modelos_panel", modelos_panel.to_dict(orient="records"))

# Pruebas H4 a H6 y parte 2 del criterio de éxito
F5, q5, p5 = wald(B_anio, INT_ANIO)
F6, q6, p6 = wald(A_reg, INT_REG)
F6b, q6b, p6b = wald(B_reg, INT_REG)
dr2 = A_sin["r2"] - A_ctrl["r2"]
h46 = pd.DataFrame([
    {"Hipótesis": "H4", "Pregunta": "¿El ICEE rural se asocia al puntaje dentro del mismo municipio? (coeficiente ≠ 0)", "Modelo": "B · efectos fijos",
     "Estadístico": f"t = {es(B['tabla'].loc['icee10', 't'])}", "gl": B["G"] - 1, "p": B["tabla"].loc["icee10", "p"]},
    {"Hipótesis": "H5", "Pregunta": "¿La asociación cambia entre 2019, 2020 y 2022? (ICEE × año)", "Modelo": "B · efectos fijos",
     "Estadístico": f"F = {es(F5)}", "gl": f"{q5} y {B_anio['G'] - 1}", "p": p5},
    {"Hipótesis": "H6", "Pregunta": "¿La asociación cambia entre regiones? (ICEE × región)", "Modelo": "A · agrupado",
     "Estadístico": f"F = {es(F6)}", "gl": f"{q6} y {A_reg['G'] - 1}", "p": p6},
])
h46["p ajustado (Holm)"] = holm(h46["p"])
h46["Decisión"] = np.where(h46["p ajustado (Holm)"] < 0.05, "Se rechaza H0", "No se rechaza H0")
guardar_tabla(h46.round(4), "A2_T19_h4_h6")
registrar("h4_h6", h46.round(4).to_dict(orient="records"))
registrar("h6_en_efectos_fijos", {"F": round(F6b, 3), "p": round(p6b, 4)})

# Pendiente del ICEE por región (modelo A con interacción)
filas = []
for r_ in ["Andina"] + [x.replace("region_", "") for x in REG]:
    pesos = {"icee10": 1.0} if r_ == "Andina" else {"icee10": 1.0, f"icee10_x_region_{r_}": 1.0}
    b_, se_, lo_, hi_, p_ = combinacion(A_reg, pesos)
    filas.append({"Región": r_, "n": int((dp["region"] == r_).sum()), "Pendiente (puntos por 10 p.p.)": b_, "Error estándar": se_,
                  "IC 95 % inf.": lo_, "IC 95 % sup.": hi_, "p": p_})
pend_reg = pd.DataFrame(filas)
pend_reg["p ajustado (Holm)"] = holm(pend_reg["p"])
pend_reg = pend_reg.round(4)
guardar_tabla(pend_reg, "A2_T20_pendiente_por_region")
registrar("pendiente_por_region", pend_reg.to_dict(orient="records"))

# Criterio de éxito completo del protocolo
p_dr2 = A_sin["tabla"].loc["icee10", "p"]
p_holm_dr2 = holm([p_dr2] + h46["p"].tolist())[0]
crit = pd.DataFrame([
    {"Parte del criterio": "|ρ| ≥ 0,10 con IC 95 % que excluye el cero en al menos 2 de 3 años (H1)",
     "Resultado": f"{anios_ok} de 3 años", "¿Se cumple?": "Sí" if anios_ok >= 2 else "No"},
    {"Parte del criterio": "ΔR² del ICEE ≥ 0,02 sobre los controles (KPI 6)", "Resultado": f"ΔR² = {es(dr2, 4)}", "¿Se cumple?": "Sí" if dr2 >= 0.02 else "No"},
    {"Parte del criterio": "… con p < 0,05 tras la corrección de Holm (errores agrupados por municipio)",
     "Resultado": f"p ajustado {p_es(p_holm_dr2) if p_holm_dr2 < 0.001 else '= ' + p_es(p_holm_dr2)}", "¿Se cumple?": "Sí" if p_holm_dr2 < 0.05 else "No"},
])
guardar_tabla(crit, "A2_T21_criterio_exito")
registrar("criterio_exito", crit.to_dict(orient="records"))
registrar("delta_r2_panel", round(float(dr2), 4))
registrar("delta_r2_p_holm", float(p_holm_dr2))
display(modelos_panel); display(h46.round(4)); display(pend_reg); display(crit)

In [ ]:
# I de Moran de los residuos del modelo A (vecindad por departamento, como en el ACA 1)
def moran_dep(valores, grupo):
    z = (valores - valores.mean()) / valores.std(ddof=0)
    s_ = z.groupby(grupo).transform("sum"); n_ = z.groupby(grupo).transform("size")
    return float((z * ((s_ - z) / (n_ - 1))).sum() / (z ** 2).sum())
dp["residuo_A"] = A["residuos"]
rng_m = np.random.default_rng(SEED + 3)
filas = []
for a_, g in dp.groupby("anio"):
    g = g[g.groupby("cod_depto")["cod_mpio"].transform("size") > 1]
    I = moran_dep(g["residuo_A"], g["cod_depto"])
    sim = [moran_dep(pd.Series(rng_m.permutation(g["residuo_A"].to_numpy()), index=g.index), g["cod_depto"]) for _ in range(999)]
    filas.append({"Año": a_, "I de Moran de los residuos del modelo A": I, "p (999 permutaciones)": (1 + sum(s >= I for s in sim)) / 1000})
moran_res = pd.DataFrame(filas).round(3)
guardar_tabla(moran_res, "A2_T22_moran_residuos_panel")
registrar("moran_residuos_panel", moran_res.to_dict(orient="records"))
display(moran_res)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.15, 1]})
mp = modelos_panel.iloc[::-1].reset_index(drop=True)
etq = [f"{m.split(' · ')[0]} · {m.split(': ')[-1] if ': ' in m else 'efectos fijos'}\n(agrupado por {g.lower()})" for m, g in
       zip(mp["Modelo"], mp["Errores agrupados por"])]
ax[0].errorbar(mp["Coef. ICEE (puntos por 10 p.p.)"], range(len(mp)),
               xerr=[mp["Coef. ICEE (puntos por 10 p.p.)"] - mp["IC 95 % inf."], mp["IC 95 % sup."] - mp["Coef. ICEE (puntos por 10 p.p.)"]],
               fmt="o", color=C1, ecolor="#8a8984", capsize=3)
ax[0].axvline(0, color="#52514e", lw=0.8)
ax[0].set_yticks(range(len(mp))); ax[0].set_yticklabels(etq, fontsize=7.5)
ax[0].set(title="Coeficiente del ICEE rural con IC 95 %", xlabel="Puntos de puntaje por cada 10 p.p. de cobertura")
pr_ = pend_reg.iloc[::-1].reset_index(drop=True)
ax[1].errorbar(pr_["Pendiente (puntos por 10 p.p.)"], range(len(pr_)),
               xerr=[pr_["Pendiente (puntos por 10 p.p.)"] - pr_["IC 95 % inf."], pr_["IC 95 % sup."] - pr_["Pendiente (puntos por 10 p.p.)"]],
               fmt="o", color=C3, ecolor="#8a8984", capsize=3)
ax[1].axvline(0, color="#52514e", lw=0.8)
ax[1].set_yticks(range(len(pr_))); ax[1].set_yticklabels(pr_["Región"])
ax[1].set(title="Pendiente del ICEE por región (modelo A, H6)", xlabel="Puntos de puntaje por cada 10 p.p. de cobertura")
fig.tight_layout(); guardar_fig(fig, "A2_F07_modelos_panel"); plt.show()

## 7. Tablero de control (primera versión del objetivo 4)
El tablero es una página HTML **autocontenida** (no necesita internet, servidor ni instalar nada: se abre con doble clic en cualquier navegador). Lo genera
este cuaderno a partir de la vista `vista_panel_rural` del modelo dimensional y de los KPIs nacionales calculados con SQL. Permite filtrar por año, región y
banda de cobertura; recalcula los KPIs con las mismas fórmulas de la sección 2.3 y muestra la lista de municipios a focalizar (cobertura baja y puntaje en
el cuartil inferior). Se guarda en `results/dashboard/tablero_aca2.html`.

In [ ]:
tab_dash = vista.merge(panel[["cod_mpio", "anio", "pct_internet"]].rename(columns={"cod_mpio": "id_municipio"}), on=["id_municipio", "anio"])
filas_dash = [{"id": r.id_municipio, "m": r.municipio, "d": r.departamento, "r": r.region, "a": int(r.anio), "n": int(r.n_estudiantes),
               "pg": float(r.punt_global), "nu": (None if pd.isna(r.n_estudiantes_urbano) else int(r.n_estudiantes_urbano)),
               "pu": (None if pd.isna(r.punt_global_urbano) else float(r.punt_global_urbano)),
               "ic": float(r.icee_rural), "vt": float(r.vt_rural), "b": int(r.id_banda), "inet": round(float(r.pct_internet), 2)}
              for r in tab_dash.itertuples()]
nacional = {str(a_): {"kpi1": round(float(kpi1[a_]), 2), "kpi2": round(float(kpi2[a_]), 2), "kpi3": round(float(kpi3[a_]), 2)} for a_ in (2019, 2020, 2022)}
rho_region = [{"g": f_["Grupo"].split(": ")[1], "rho": round(f_["ρ de Spearman"], 3), "lo": round(f_["IC 95 % inf."], 3),
               "hi": round(f_["IC 95 % sup."], 3), "sig": f_["Decisión"] == "Se rechaza H0"} for _, f_ in h1.iterrows() if f_["Grupo"].startswith("Región")]
meta = {"generado": dt.date.today().isoformat(), "version": "v0.2", "filas": len(filas_dash)}
PAYLOAD = json.dumps({"filas": filas_dash, "nacional": nacional, "rho_region": rho_region, "meta": meta}, ensure_ascii=False, separators=(",", ":"))

HTML = r"""<!doctype html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Tablero energía rural y Saber 11</title>
<style>
:root{--bg:#f7f6f2;--panel:#ffffff;--ink:#1d2433;--mut:#5f6673;--line:#e2e0d8;--azul:#2a78d6;--naranja:#eb6834;--verde:#1baf7a;--gris:#8a8984;
--baja:#eb6834;--media:#e0a526;--alta:#2a78d6;--sombra:0 1px 2px rgba(20,20,20,.06)}
@media (prefers-color-scheme: dark){:root:not([data-theme="light"]){--bg:#15171c;--panel:#1e2128;--ink:#e9e7e1;--mut:#a3a7b0;--line:#30343d;--sombra:none}}
:root[data-theme="dark"]{--bg:#15171c;--panel:#1e2128;--ink:#e9e7e1;--mut:#a3a7b0;--line:#30343d;--sombra:none}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:14px/1.45 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
header{padding:20px 24px 8px;max-width:1280px;margin:auto}h1{font-size:22px;margin:0 0 4px}header p{margin:0;color:var(--mut)}
.filtros{display:flex;flex-wrap:wrap;gap:10px;align-items:end;max-width:1280px;margin:8px auto 0;padding:0 24px}
.filtros label{display:flex;flex-direction:column;font-size:12px;color:var(--mut);gap:3px}
select,input{font:inherit;color:var(--ink);background:var(--panel);border:1px solid var(--line);border-radius:6px;padding:6px 8px;min-width:150px}
button{font:inherit;background:var(--panel);color:var(--ink);border:1px solid var(--line);border-radius:6px;padding:6px 12px;cursor:pointer}
main{max-width:1280px;margin:auto;padding:14px 24px 30px}
.kpis{display:grid;grid-template-columns:repeat(6,minmax(0,1fr));gap:10px}
.kpi{background:var(--panel);border:1px solid var(--line);border-radius:10px;padding:12px 14px;box-shadow:var(--sombra)}
.kpi .t{font-size:12px;color:var(--mut)}.kpi .v{font-size:26px;font-weight:650;margin:2px 0;font-variant-numeric:tabular-nums}.kpi .s{font-size:11.5px;color:var(--mut)}
.grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;margin-top:12px}
.card{background:var(--panel);border:1px solid var(--line);border-radius:10px;padding:12px 14px;box-shadow:var(--sombra);min-width:0}
.card h2{font-size:14.5px;margin:0 0 2px}.card .sub{font-size:12px;color:var(--mut);margin:0 0 6px}
svg{width:100%;height:auto;display:block}svg text{fill:var(--ink);font-size:11px}svg .eje{stroke:var(--line)}svg .mut{fill:var(--mut)}
.leyenda{display:flex;gap:14px;font-size:12px;color:var(--mut);margin-top:4px;flex-wrap:wrap}.leyenda i{display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:5px;vertical-align:-1px}
table{width:100%;border-collapse:collapse;font-size:12.5px}th,td{text-align:left;padding:5px 6px;border-bottom:1px solid var(--line)}
th{color:var(--mut);font-weight:600}td.n{text-align:right;font-variant-numeric:tabular-nums}.tabla{max-height:340px;overflow:auto}
footer{max-width:1280px;margin:auto;padding:0 24px 28px;color:var(--mut);font-size:12px}
@media (max-width:900px){.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.grid{grid-template-columns:1fr}header,.filtros,main,footer{padding-left:16px;padding-right:16px}}
</style></head><body>
<header><h1>Cobertura eléctrica rural y rendimiento en Saber 11</h1>
<p>Municipios con estudiantes de sedes rurales, 2019, 2020 y 2022 · ICFES y UPME · Proyecto de grado CUN, Avance 2 (v0.2)</p></header>
<div class="filtros">
<label>Año<select id="fAnio"><option value="">Todos</option><option>2019</option><option>2020</option><option>2022</option></select></label>
<label>Región<select id="fReg"><option value="">Todas</option><option>Amazonía</option><option>Andina</option><option>Caribe</option><option>Orinoquía</option><option>Pacífica</option></select></label>
<label>Banda de cobertura rural<select id="fBanda"><option value="">Todas</option><option value="1">Baja (&lt;70 %)</option><option value="2">Media (70–95 %)</option><option value="3">Alta (≥95 %)</option></select></label>
<button id="limpiar">Quitar filtros</button>
</div>
<main>
<section class="kpis" id="kpis"></section>
<section class="grid">
<div class="card"><h2>KPI 4 · Puntaje rural por banda de cobertura</h2><p class="sub">Promedio de los puntajes municipales; n = municipio–año</p><div id="gBandas"></div></div>
<div class="card"><h2>Cobertura y puntaje por municipio</h2><p class="sub">Cada punto es un municipio en un año; pase el cursor para ver el nombre</p><div id="gDisp"></div>
<div class="leyenda"><span><i style="background:var(--baja)"></i>Baja</span><span><i style="background:var(--media)"></i>Media</span><span><i style="background:var(--alta)"></i>Alta</span></div></div>
<div class="card"><h2>KPI 1 y KPI 2 · Puntaje rural y urbano por año</h2><p class="sub">Municipios de la selección (sin filtro de año); la distancia entre líneas es la brecha</p><div id="gTend"></div>
<div class="leyenda"><span><i style="background:var(--azul)"></i>Urbano</span><span><i style="background:var(--naranja)"></i>Rural</span></div></div>
<div class="card"><h2>KPI 5 · Asociación por región (prueba H1)</h2><p class="sub">ρ de Spearman con IC 95 %; sólido = significativo tras Holm (2019–2022)</p><div id="gRho"></div></div>
</section>
<section class="card" style="margin-top:12px"><h2>Municipios para focalizar</h2>
<p class="sub">Cobertura rural baja (&lt;70 %) y puntaje en el cuartil inferior de la selección · <span id="nFoco"></span></p>
<input id="buscar" placeholder="Buscar municipio o departamento" style="margin-bottom:8px;width:min(360px,100%)">
<div class="tabla"><table><thead><tr><th>Municipio</th><th>Departamento</th><th>Región</th><th>Año</th><th class="n">ICEE rural</th><th class="n">Puntaje rural</th><th class="n">Estudiantes</th><th class="n">% con internet</th></tr></thead><tbody id="tFoco"></tbody></table></div>
</section>
</main>
<footer>KPI 1: promedio del puntaje global de los estudiantes rurales (ponderado por estudiantes). KPI 2: puntaje urbano − rural en los municipios con ambas zonas.
KPI 3: 100 × viviendas rurales con servicio / viviendas rurales. KPI 4: puntaje medio municipal banda Alta − banda Baja. KPI 5: ρ de Spearman entre ICEE rural y puntaje.
Muestra asociaciones, no efectos causales. Fuentes: ICFES (Saber 11, CC BY-SA 4.0) y UPME (ICEE). Generado el <span id="fecha"></span> por el cuaderno ACA2.</footer>
<script>
const D = __DATOS__;
const $ = id => document.getElementById(id);
const fmt = (x, d=1) => x == null || isNaN(x) ? "—" : x.toLocaleString("es-CO", {minimumFractionDigits:d, maximumFractionDigits:d});
const COL = {1:"var(--baja)", 2:"var(--media)", 3:"var(--alta)"}, NOMB = {1:"Baja", 2:"Media", 3:"Alta"};
function rangos(v){const o=v.map((x,i)=>[x,i]).sort((a,b)=>a[0]-b[0]);const r=new Array(v.length);let i=0;
 while(i<o.length){let j=i;while(j+1<o.length&&o[j+1][0]===o[i][0])j++;const m=(i+j)/2+1;for(let k=i;k<=j;k++)r[o[k][1]]=m;i=j+1;}return r;}
function pearson(x,y){const n=x.length;const mx=x.reduce((a,b)=>a+b,0)/n,my=y.reduce((a,b)=>a+b,0)/n;let sxy=0,sx=0,sy=0;
 for(let i=0;i<n;i++){sxy+=(x[i]-mx)*(y[i]-my);sx+=(x[i]-mx)**2;sy+=(y[i]-my)**2;}return sxy/Math.sqrt(sx*sy);}
const spearman = (x,y) => x.length < 3 ? NaN : pearson(rangos(x), rangos(y));
const media = a => a.length ? a.reduce((s,x)=>s+x,0)/a.length : NaN;
function seleccion(conAnio=true){const a=$("fAnio").value,r=$("fReg").value,b=$("fBanda").value;
 return D.filas.filter(f=>(!conAnio||!a||f.a==a)&&(!r||f.r==r)&&(!b||f.b==b));}
function kpis(s){const n=s.reduce((t,f)=>t+f.n,0);const k1=s.reduce((t,f)=>t+f.pg*f.n,0)/n;
 const u=s.filter(f=>f.nu);const nu=u.reduce((t,f)=>t+f.nu,0),nr=u.reduce((t,f)=>t+f.n,0);
 const k2=u.reduce((t,f)=>t+f.pu*f.nu,0)/nu-u.reduce((t,f)=>t+f.pg*f.n,0)/nr;
 const vt=s.reduce((t,f)=>t+f.vt,0);const k3=s.reduce((t,f)=>t+f.vt*f.ic/100,0)/vt*100;
 const k4=media(s.filter(f=>f.b==3).map(f=>f.pg))-media(s.filter(f=>f.b==1).map(f=>f.pg));
 const k5=spearman(s.map(f=>f.ic),s.map(f=>f.pg));return {n,k1,k2,k3,k4,k5,mun:new Set(s.map(f=>f.id)).size,filas:s.length};}
function tarjetas(k){const a=$("fAnio").value||"2019";const nac=D.nacional[a];
 const T=[["KPI 1 · Puntaje global rural",fmt(k.k1,1),`Nacional ${a}: ${fmt(nac.kpi1,1)} · ponderado por estudiantes`],
 ["KPI 2 · Brecha urbano–rural",fmt(k.k2,1)+" pts",`Nacional ${a}: ${fmt(nac.kpi2,1)} pts`],
 ["KPI 3 · ICEE rural",fmt(k.k3,1)+" %",`Nacional ${a}: ${fmt(nac.kpi3,1)} % · ponderado por viviendas`],
 ["KPI 4 · Diferencia Alta − Baja",isNaN(k.k4)?"—":fmt(k.k4,1)+" pts","Línea base: 0 puntos"],
 ["KPI 5 · ρ de Spearman",isNaN(k.k5)?"—":fmt(k.k5,3),"Línea base: ρ = 0"],
 ["Municipios y estudiantes",k.mun.toLocaleString("es-CO"),`${k.n.toLocaleString("es-CO")} estudiantes rurales · ${k.filas.toLocaleString("es-CO")} municipio–año`]];
 $("kpis").innerHTML=T.map(t=>`<div class="kpi"><div class="t">${t[0]}</div><div class="v">${t[1]}</div><div class="s">${t[2]}</div></div>`).join("");}
function svg(w,h,c){return `<svg viewBox="0 0 ${w} ${h}" role="img">${c}</svg>`;}
function barrasBanda(s){const W=520,H=230,m={l:40,r:10,t:18,b:34};const v=[1,2,3].map(b=>{const g=s.filter(f=>f.b==b);return {b,y:media(g.map(f=>f.pg)),n:g.length};});
 const lo=190,hi=250;const x=i=>m.l+i*(W-m.l-m.r)/3+18,bw=(W-m.l-m.r)/3-36,y=t=>H-m.b-(Math.max(lo,Math.min(hi,t))-lo)/(hi-lo)*(H-m.t-m.b);
 let c="";for(let t=lo;t<=hi;t+=10)c+=`<line class="eje" x1="${m.l}" x2="${W-m.r}" y1="${y(t)}" y2="${y(t)}"/><text class="mut" x="${m.l-6}" y="${y(t)+4}" text-anchor="end">${t}</text>`;
 v.forEach((d,i)=>{if(!d.n){c+=`<text class="mut" x="${x(i)+bw/2}" y="${H-m.b-8}" text-anchor="middle">sin datos</text>`;}else{
 c+=`<rect x="${x(i)}" y="${y(d.y)}" width="${bw}" height="${H-m.b-y(d.y)}" fill="${COL[d.b]}" rx="3"><title>${NOMB[d.b]}: ${fmt(d.y,2)} (n = ${d.n})</title></rect>
 <text x="${x(i)+bw/2}" y="${y(d.y)-5}" text-anchor="middle" font-weight="600">${fmt(d.y,1)}</text>`;}
 c+=`<text x="${x(i)+bw/2}" y="${H-m.b+15}" text-anchor="middle">${NOMB[d.b]}</text><text class="mut" x="${x(i)+bw/2}" y="${H-m.b+28}" text-anchor="middle">n = ${d.n}</text>`;});
 c+=`<text class="mut" x="${m.l}" y="11">Eje desde 190 puntos</text>`;$("gBandas").innerHTML=svg(W,H,c);}
function dispersion(s){const W=520,H=250,m={l:40,r:12,t:10,b:32};const x=t=>m.l+t/100*(W-m.l-m.r),y=t=>H-m.b-(t-140)/(350-140)*(H-m.t-m.b);
 let c="";for(let t=150;t<=350;t+=50)c+=`<line class="eje" x1="${m.l}" x2="${W-m.r}" y1="${y(t)}" y2="${y(t)}"/><text class="mut" x="${m.l-6}" y="${y(t)+4}" text-anchor="end">${t}</text>`;
 for(let t=0;t<=100;t+=20)c+=`<text class="mut" x="${x(t)}" y="${H-m.b+15}" text-anchor="middle">${t}</text>`;
 c+=`<text class="mut" x="${(W+m.l)/2}" y="${H-3}" text-anchor="middle">ICEE rural (%)</text>`;
 s.forEach(f=>{c+=`<circle cx="${x(f.ic).toFixed(1)}" cy="${y(f.pg).toFixed(1)}" r="2.6" fill="${COL[f.b]}" fill-opacity=".55"><title>${f.m} (${f.d}), ${f.a}: ICEE ${fmt(f.ic,1)} %, puntaje ${fmt(f.pg,1)}</title></circle>`;});
 $("gDisp").innerHTML=svg(W,H,c);}
function tendencia(s){const W=520,H=220,m={l:40,r:70,t:14,b:28};const A=[2019,2020,2022];
 const p=A.map(a=>{const g=s.filter(f=>f.a==a);const n=g.reduce((t,f)=>t+f.n,0);const u=g.filter(f=>f.nu);const nu=u.reduce((t,f)=>t+f.nu,0);
 return {a,r:g.reduce((t,f)=>t+f.pg*f.n,0)/n,u:u.reduce((t,f)=>t+f.pu*f.nu,0)/nu};});
 const lo=200,hi=280;const x=a=>m.l+(a-2019)/3*(W-m.l-m.r),y=t=>H-m.b-(t-lo)/(hi-lo)*(H-m.t-m.b);let c="";
 for(let t=lo;t<=hi;t+=20)c+=`<line class="eje" x1="${m.l}" x2="${W-m.r}" y1="${y(t)}" y2="${y(t)}"/><text class="mut" x="${m.l-6}" y="${y(t)+4}" text-anchor="end">${t}</text>`;
 A.forEach(a=>c+=`<text class="mut" x="${x(a)}" y="${H-m.b+16}" text-anchor="middle">${a}</text>`);
 [["u","var(--azul)","Urbano"],["r","var(--naranja)","Rural"]].forEach(([k,col,et])=>{const pts=p.filter(q=>!isNaN(q[k]));
 c+=`<polyline fill="none" stroke="${col}" stroke-width="2.2" points="${pts.map(q=>x(q.a)+","+y(q[k])).join(" ")}"/>`;
 pts.forEach(q=>c+=`<circle cx="${x(q.a)}" cy="${y(q[k])}" r="3.5" fill="${col}"><title>${et} ${q.a}: ${fmt(q[k],1)}</title></circle><text x="${x(q.a)}" y="${y(q[k])-8}" text-anchor="middle" font-size="10">${fmt(q[k],1)}</text>`);
 if(pts.length)c+=`<text x="${x(pts[pts.length-1].a)+10}" y="${y(pts[pts.length-1][k])+4}">${et}</text>`;});
 p.forEach(q=>{if(!isNaN(q.u)&&!isNaN(q.r))c+=`<text class="mut" x="${x(q.a)+4}" y="${(y(q.u)+y(q.r))/2+4}" font-size="10">brecha ${fmt(q.u-q.r,1)}</text>`;});
 $("gTend").innerHTML=svg(W,H,c);}
function rhoRegion(){const W=520,H=210,m={l:80,r:20,t:8,b:30};const R=D.rho_region,sel=$("fReg").value;
 const x=t=>m.l+(t+0.2)/0.8*(W-m.l-m.r),y=i=>m.t+i*(H-m.t-m.b)/R.length+12;let c="";
 for(let t=-0.2;t<=0.61;t+=0.2)c+=`<line class="eje" x1="${x(t)}" x2="${x(t)}" y1="${m.t}" y2="${H-m.b}"/><text class="mut" x="${x(t)}" y="${H-m.b+15}" text-anchor="middle">${fmt(t,1)}</text>`;
 c+=`<line x1="${x(0)}" x2="${x(0)}" y1="${m.t}" y2="${H-m.b}" stroke="var(--gris)"/>`;
 R.forEach((d,i)=>{const op=!sel||sel==d.g?1:.3;c+=`<g opacity="${op}"><text x="${m.l-8}" y="${y(i)+4}" text-anchor="end">${d.g}</text>
 <line x1="${x(d.lo)}" x2="${x(d.hi)}" y1="${y(i)}" y2="${y(i)}" stroke="var(--gris)" stroke-width="1.5"/>
 <circle cx="${x(d.rho)}" cy="${y(i)}" r="5" fill="${d.sig?"var(--verde)":"var(--panel)"}" stroke="var(--verde)" stroke-width="2"><title>${d.g}: ρ = ${fmt(d.rho,3)} (IC ${fmt(d.lo,3)} a ${fmt(d.hi,3)})</title></circle>
 <text class="mut" x="${x(d.hi)+6}" y="${y(i)+4}">${fmt(d.rho,2)}</text></g>`;});
 $("gRho").innerHTML=svg(W,H,c);}
function foco(s){const q=s.map(f=>f.pg).sort((a,b)=>a-b);const q1=q.length?q[Math.floor(0.25*(q.length-1))]:0;const txt=$("buscar").value.toLowerCase();
 const f=s.filter(f=>f.ic<70&&f.pg<=q1).filter(f=>!txt||(f.m+" "+f.d).toLowerCase().includes(txt)).sort((a,b)=>a.pg-b.pg);
 $("nFoco").textContent=`${f.length} municipio–año (umbral del cuartil: ${fmt(q1,1)} puntos)`;
 $("tFoco").innerHTML=f.map(f=>`<tr><td>${f.m}</td><td>${f.d}</td><td>${f.r}</td><td>${f.a}</td><td class="n">${fmt(f.ic,1)} %</td><td class="n">${fmt(f.pg,1)}</td><td class="n">${f.n}</td><td class="n">${fmt(f.inet,1)} %</td></tr>`).join("");}
function actualizar(){const s=seleccion();tarjetas(kpis(s));barrasBanda(s);dispersion(s);tendencia(seleccion(false));rhoRegion();foco(s);}
["fAnio","fReg","fBanda"].forEach(id=>$(id).addEventListener("change",actualizar));$("buscar").addEventListener("input",()=>foco(seleccion()));
$("limpiar").addEventListener("click",()=>{["fAnio","fReg","fBanda","buscar"].forEach(id=>$(id).value="");actualizar();});
$("fecha").textContent=D.meta.generado;actualizar();
</script></body></html>"""
(DASH / "tablero_aca2.html").write_text(HTML.replace("__DATOS__", PAYLOAD), encoding="utf-8")
print("Tablero:", (DASH / "tablero_aca2.html").relative_to(ROOT), f"({(DASH / 'tablero_aca2.html').stat().st_size / 1024:.0f} KB)")

# Validación de totales del tablero contra el modelo dimensional (mismas fórmulas que usa la página, sin filtros)
fd = pd.DataFrame(filas_dash)
u_ = fd[fd["nu"].notna()]
val_dash = pd.DataFrame([
    {"Control": "Filas municipio–año", "Modelo (SQL)": len(vista), "Tablero (datos de la página)": len(fd)},
    {"Control": "Municipios", "Modelo (SQL)": vista["id_municipio"].nunique(), "Tablero (datos de la página)": fd["id"].nunique()},
    {"Control": "Estudiantes rurales", "Modelo (SQL)": int(vista["n_estudiantes"].sum()), "Tablero (datos de la página)": int(fd["n"].sum())},
    {"Control": "KPI 4 · Diferencia Alta − Baja", "Modelo (SQL)": round(kpi4_total, 4),
     "Tablero (datos de la página)": round(fd.loc[fd.b == 3, "pg"].mean() - fd.loc[fd.b == 1, "pg"].mean(), 4)},
    {"Control": "KPI 5 · ρ de Spearman", "Modelo (SQL)": round(kpi5_total, 4), "Tablero (datos de la página)": round(stats.spearmanr(fd["ic"], fd["pg"]).statistic, 4)},
])
val_dash["Diferencia"] = (val_dash["Tablero (datos de la página)"] - val_dash["Modelo (SQL)"]).round(6)
guardar_tabla(val_dash, "A2_T23_validacion_tablero")
registrar("validacion_tablero", val_dash.to_dict(orient="records"))
assert (val_dash["Diferencia"].abs() < 1e-9).all()
# Cifras que la página muestra sin filtros (sirven para comprobar las capturas)
registrar("tablero_sin_filtros", {"kpi1": round(float((fd.pg * fd.n).sum() / fd.n.sum()), 2),
                                  "kpi2": round(float((u_.pu * u_.nu).sum() / u_.nu.sum() - (u_.pg * u_.n).sum() / u_.n.sum()), 2),
                                  "kpi3": round(float((fd.vt * fd.ic / 100).sum() / fd.vt.sum() * 100), 2),
                                  "kpi4": round(float(fd.loc[fd.b == 3, "pg"].mean() - fd.loc[fd.b == 1, "pg"].mean()), 2),
                                  "kpi5": round(float(stats.spearmanr(fd["ic"], fd["pg"]).statistic), 3)})
q1_ = fd["pg"].quantile(0.25, interpolation="lower")
registrar("tablero_municipios_foco", int(((fd["ic"] < 70) & (fd["pg"] <= q1_)).sum()))
display(val_dash)

### 7.1 Tablas para Power BI (alternativa al tablero HTML)
Si se prefiere construir el tablero en Power BI, estas tablas reproducen el esquema estrella en archivos livianos. Los hechos de Saber 11 se entregan
agregados por municipio, año y zona con **sumas** (no promedios), para que Power BI calcule promedios ponderados exactos: puntaje promedio =
suma de puntajes / número de estudiantes. Se guardan en `data/processed/powerbi/`.

In [ ]:
PBI = PROCESSED / "powerbi"
PBI.mkdir(parents=True, exist_ok=True)
agregado = sql("""SELECT id_municipio, anio, id_zona, COUNT(*) AS n_estudiantes, SUM(punt_global) AS suma_punt_global,
                         SUM(punt_matematicas) AS suma_punt_matematicas, SUM(punt_lectura) AS suma_punt_lectura,
                         SUM(internet) AS n_con_internet, COUNT(internet) AS n_responde_internet,
                         SUM(computador) AS n_con_computador, COUNT(computador) AS n_responde_computador
                  FROM hechos_saber11 GROUP BY id_municipio, anio, id_zona""")
tablas_pbi = {"dim_municipio": sql("SELECT * FROM dim_municipio"), "dim_tiempo": sql("SELECT * FROM dim_tiempo"),
              "dim_zona": sql("SELECT * FROM dim_zona"), "dim_banda_icee": sql("SELECT * FROM dim_banda_icee"),
              "hechos_cobertura": sql("SELECT * FROM hechos_cobertura"), "hechos_saber11_agregado": agregado,
              "panel_rural": tab_dash.drop(columns=["etapa_pandemia"]), "kpi_asociacion_region": h1[["Grupo", "n", "ρ de Spearman", "IC 95 % inf.", "IC 95 % sup.", "p ajustado (Holm)", "Decisión"]]}
for nombre, tabla in tablas_pbi.items():
    tabla.to_csv(PBI / f"{nombre}.csv", index=False, encoding="utf-8-sig", sep=";", decimal=",")
assert int(agregado["n_estudiantes"].sum()) == len(hechos_saber11)
assert abs(agregado["suma_punt_global"].sum() - hechos_saber11["punt_global"].sum()) < 1e-6
# Cifras de control para comprobar el tablero de Power BI
control_pbi = pd.DataFrame([
    {"Control": "Estudiantes en hechos_saber11_agregado", "Valor esperado": int(agregado["n_estudiantes"].sum())},
    {"Control": "Filas de panel_rural", "Valor esperado": len(tab_dash)},
    {"Control": "Estudiantes rurales en panel_rural", "Valor esperado": int(tab_dash["n_estudiantes"].sum())},
] + [{"Control": f"KPI 1 puntaje rural {a_}", "Valor esperado": round(float(kpi1[a_]), 2)} for a_ in (2019, 2020, 2022)]
  + [{"Control": f"KPI 2 brecha urbano–rural {a_}", "Valor esperado": round(float(kpi2[a_]), 2)} for a_ in (2019, 2020, 2022)]
  + [{"Control": f"KPI 3 ICEE rural nacional {a_}", "Valor esperado": round(float(kpi3[a_]), 2)} for a_ in (2019, 2020, 2022)]
  + [{"Control": "KPI 4 diferencia Alta − Baja (panel, 2019–2022)", "Valor esperado": round(float(kpi4_total), 2)}])
guardar_tabla(control_pbi, "A2_T25_controles_powerbi")
registrar("controles_powerbi", control_pbi.to_dict(orient="records"))
print("Tablas para Power BI:", ", ".join(f"{k} ({len(v):,} filas)" for k, v in tablas_pbi.items()))
display(control_pbi)

## 8. Trazabilidad y salidas
Se agrega a `results/experiments.csv` **una fila por modelo y corrida** (fecha, modelo, parámetros, semilla y métricas), se guardan todas las cifras
en `results/tables/cifras_aca2.json` y se actualiza la tarjeta del modelo. Las decisiones nuevas están en `docs/bitacora_decisiones.csv`.

In [ ]:
ahora = dt.datetime.now().isoformat(timespec="seconds")
entorno = f"{platform.system()} {platform.machine()} (CPU) · Python {platform.python_version()} · scikit-learn {sklearn.__version__}"
filas_exp = []
for _, f_ in comparacion.iterrows():
    filas_exp.append({"fecha": ahora, "aca": "ACA2", "modelo": f_["Modelo"],
                      "parametros": json.dumps({k: (v if isinstance(v, (str, int, float, type(None), list, tuple)) else float(v))
                                                for k, v in mejores[f_["Modelo"]].items()}, ensure_ascii=False, default=str),
                      "semilla": SEED, "entorno": entorno, "tiempo_s": round(float(f_["Tiempo medio de entrenamiento (s)"]), 3),
                      "metricas": json.dumps({"cv_rmse_media": round(f_["RMSE media"], 4), "cv_rmse_de": round(f_["RMSE d.e."], 4),
                                              "cv_mae_media": round(f_["MAE media"], 4), "cv_mape_media": round(f_["MAPE % media"], 4),
                                              "cv_r2_media": round(f_["R² media"], 4), "prueba_rmse": round(f_["prueba_rmse"], 4),
                                              "prueba_mae": round(f_["prueba_mae"], 4), "prueba_r2": round(f_["prueba_r2"], 4),
                                              "particiones": len(CV), "variables": len(X_COLS)}, ensure_ascii=False)})
for _, f_ in modelos_panel.iterrows():
    filas_exp.append({"fecha": ahora, "aca": "ACA2", "modelo": f"Panel · {f_['Modelo']} (EE por {f_['Errores agrupados por'].lower()})",
                      "parametros": json.dumps({"controles": CTRL}, ensure_ascii=False), "semilla": SEED, "entorno": entorno, "tiempo_s": "",
                      "metricas": json.dumps({"coef_icee_10pp": round(f_["Coef. ICEE (puntos por 10 p.p.)"], 4), "p": round(f_["p"], 6),
                                              "r2": round(f_["R²"], 4), "n": int(f_["Observaciones"])}, ensure_ascii=False)})
exp_path = ROOT / "results/experiments.csv"
previo = pd.read_csv(exp_path, dtype=str) if exp_path.exists() else pd.DataFrame()
previo = previo[previo.get("aca", pd.Series(dtype=str)) != "ACA2"] if len(previo) else previo   # una sola corrida vigente del ACA 2
pd.concat([previo, pd.DataFrame(filas_exp).astype(str)], ignore_index=True).to_csv(exp_path, index=False, encoding="utf-8")
registrar("experimentos_filas_aca2", len(filas_exp))

mejor = comparacion[comparacion["Modelo"] == MEJOR].iloc[0]
base_ = comparacion[comparacion["Modelo"] == "Línea base (promedio)"].iloc[0]
energia = contexto[contexto["Grupo quitado"] == "Energía"].iloc[0]
tarjeta = f"""# Tarjeta del modelo (versión v0.2, ACA 2)

**Propósito.** Diagnóstico, no predicción: medir cuánta variación del puntaje global rural de Saber 11 (promedio municipal, 2019, 2020 y 2022) se asocia a
la cobertura eléctrica rural (ICEE de la UPME), después de tener en cuenta el contexto socioeconómico, territorial y escolar (objetivo específico 3).

**Datos.** Panel municipal rural de {len(d):,} municipio–año ({d['cod_mpio'].nunique()} municipios). Prueba: 20 % de los municipios, separados antes de
cualquier transformación. Variables: {len(X_COLS)} seleccionadas con LASSO (estabilidad ≥ 80 % en 15 particiones) e información mutua.

**Modelos comparados.** {', '.join(MODELOS)}; además stacking y votación de los tres mejores. Validación: 5 pliegues × 3 repeticiones, agrupados por
municipio y estratificados por banda de cobertura; búsqueda aleatoria de {N_PRUEBAS} combinaciones por modelo.

**Mejor modelo: {MEJOR}.** RMSE de validación {mejor['RMSE media']:.2f} ± {mejor['RMSE d.e.']:.2f} puntos (línea base {base_['RMSE media']:.2f});
R² {mejor['R² media']:.3f} ± {mejor['R² d.e.']:.3f}; en la prueba RMSE {mejor['prueba_rmse']:.2f} y R² {mejor['prueba_r2']:.3f}. Sin las variables de energía
el RMSE sube {energia['Aumento del RMSE (media)']:.2f} ± {energia['Aumento del RMSE (d.e.)']:.2f} puntos.

**Modelo inferencial.** Panel con efectos fijos de municipio y año y errores agrupados por municipio (H4–H6); ver `results/tables/A2_T18_modelos_panel.csv`.

**Limitaciones.** Asociación, no causalidad. Unidad municipal (no se concluye sobre estudiantes). El ICEE mide cobertura, no calidad del servicio.
Pocos años por municipio (máximo 3). Usos no recomendados: asignar recursos a un municipio solo con la predicción del modelo.
"""
(ROOT / "models/model_card.md").write_text(tarjeta, encoding="utf-8")

CIFRAS["fecha_ejecucion"] = ahora
CIFRAS["semilla"] = SEED
CIFRAS["tiempo_ejecucion_s"] = round(time.time() - T0, 1)
json.dump(CIFRAS, open(TAB / "cifras_aca2.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
con.close()
print(f"Listo en {time.time() - T0:.0f} s. Mejor modelo: {MEJOR}. Cifras: results/tables/cifras_aca2.json")

# En Google Colab: empaqueta los resultados y los descarga, para subirlos al repositorio (results/, models/ y el modelo dimensional en CSV)
if EN_COLAB:
    import shutil, zipfile
    from google.colab import files
    with zipfile.ZipFile("resultados_ACA2.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for carpeta in ["results", "models", "data/processed/modelo_dimensional"]:
            for f in (ROOT / carpeta).rglob("*"):
                if f.is_file():
                    z.write(f, f.relative_to(ROOT))
    files.download("resultados_ACA2.zip")